# Original Task 2 execution history
This notebook retains the original cell source. Outputs and private notebook metadata were removed for publication. It contains retired setup cells and repeated resume cells. Follow workflow/README.md and run one applicable cell at a time. The CPU reproduction notebook is the supported entry point for checking the published results.

In [ ]:
# Block 1: CPU-only setup and persistent approval records.
import os, sys, json, uuid, shutil, hashlib, platform, subprocess
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from importlib import metadata
from google.colab import drive, runtime

# Release an accidentally allocated accelerator before this CPU-only task.
gpu_detected = os.environ.get("COLAB_GPU", "0") not in ("", "0")
if shutil.which("nvidia-smi"):
    probe = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    )
    gpu_detected = gpu_detected or (
        probe.returncode == 0 and bool(probe.stdout.strip())
    )

if gpu_detected or os.environ.get("COLAB_TPU_ADDR"):
    print("Accelerator detected. CPU setup needs no GPU. Requesting release.",
          flush=True)
    try:
        runtime.unassign()
    finally:
        raise RuntimeError(
            "Select Hardware accelerator > None, then rerun this block."
        )

# Authorize Google Drive when Colab prompts you.
drive.mount("/content/drive")
my_drive = Path("/content/drive/MyDrive")
if not my_drive.is_dir():
    raise RuntimeError("MyDrive was not found. Stop and share the output.")

# Unique folders preserve existing work, including if this cell is rerun.
started = datetime.now(ZoneInfo("Asia/Karachi"))
run_id = started.strftime("run_%Y%m%d_%H%M%S_") + uuid.uuid4().hex[:8]

TASK2_ROOT = my_drive / "ATML_PA3" / "task2" / run_id
TASK2_ROOT.mkdir(parents=True, exist_ok=False)

for folder in (
    "provenance", "configs", "manifests", "logs", "checkpoints", "results"
):
    (TASK2_ROOT / folder).mkdir(exist_ok=False)

TASK2_LOCAL = Path("/content/atml_pa3_task2") / run_id
TASK2_LOCAL.mkdir(parents=True, exist_ok=False)

# Preserve approved directions and any conditions still needing specification.
approved_directions = [
    (1, "Scope",
     "Mandatory 196-token run first; defer optional compression."),
    (2, "Storage",
     "Fresh Drive workspace for durable checkpoints/results; local Colab disk "
     "for active reads."),
    (3, "GPU segments",
     "At most 250 training updates per block; scheduled evaluation and "
     "verified durable saving before runtime release."),
    (4, "Error handling",
     "Stop on handled errors; retain diagnostics and prior verified checkpoint; "
     "bounded save attempt followed by release. Exact timeout remains open."),
    (5, "Data selection",
     "Sort IDs before seeded selection; jointly allocate image-disjoint "
     "evaluation suites; preserve exact IDs and orders."),
    (6, "Caption selection and ordering",
     "Three distinct eligible original training captions per image, one "
     "development caption; two seeded shuffled training passes."),
    (7, "Grounding evaluation allocation",
     "Development: 12 presence and 12 counting pairs. "
     "Final test: 32 presence and 32 counting pairs."),
    (8, "Grounding image universe",
     "Initially use the union of selected caption/natural-VQA training images, "
     "subject to category/count feasibility; review a revision if insufficient."),
    (9, "Grounding sampling",
     "Uniform eligible category selection, then eligible pairs; sample "
     "different count combinations between 1 and 4."),
    (10, "Annotation quality",
     "Exclude crowd-ambiguous counting candidates; inspect all grounding "
     "evaluation pairs and a seeded training sample independently of model "
     "predictions. Exact remaining audit details are open."),
    (11, "Natural answer targets",
     "Official VQA normalization; choose the original consensus answer with "
     "lowest annotation-answer ID; retain all ten references."),
    (12, "Content caps",
     "Prefer complete eligible examples within prescribed caps over truncation; "
     "verify that every required quota remains feasible."),
    (13, "Prompts and chat format",
     "Fixed caption/short-answer instructions within native Qwen chat structure. "
     "Show exact text before tokenization; no additional trainable image markers "
     "or unapproved instructions."),
    (14, "Connector initialization and calibration",
     "Seeded standard Linear initialization including biases; fixed training-only "
     "RMS calibration. Exact sample, formula, and placement remain to specify."),
    (15, "LoRA initialization",
     "Standard random A / zero B; preserve manual-prescribed adapter settings."),
    (16, "AdamW details",
     "Betas (0.9, 0.999), epsilon 1e-8, weight decay 0.01 on all "
     "trainable parameters."),
    (17, "Stage transitions",
     "Retain connector optimizer moments across stages and LoRA moments from "
     "Stages 2 to 3; restart per-stage schedules; preserve scaler state."),
    (18, "Warmup rounding",
     "113, 75, 38 updates. Exact first/warmup-end/final LR conventions "
     "must be reviewable before implementation."),
    (19, "Numerical failures",
     "Stop/report nonfinite loss/gradients or skipped updates; no silent changes "
     "to precision, clipping, batch size, sampler, or budget. "
     "Agree recovery details if failure occurs."),
    (20, "Evaluation conventions",
     "Seeded frozen mismatched mapping; generation timing excludes model loading; "
     "percentile pair-bootstrap intervals with seed 6304. Additional development "
     "generation needs a separate decision."),
    (21, "Qualitative selection",
     "Reproducible coverage of successes, failures, and image-swap disagreements; "
     "exact categories/counts remain to specify before final analysis."),
    (22, "Software and publication",
     "Pin compatible libraries/model revisions after checks; publication requires "
     "separate authorization; no silent recipe or acceleration changes."),
]

approvals = {
    "assignment": "ATML PA3 Task 2",
    "approval_date": "2026-10-31",
    "recorded_in_colab_at": started.isoformat(),
    "timezone": "Asia/Karachi",
    "approval_scope": "All 22 proposed choices, with their stated conditions",
    "user_approval": (
        "Approved, as long as you think they're sensible within the context "
        "of what we're doing, and make sure to log and save all of these "
        "for the report later"
    ),
    "clarification": (
        "User confirmed this approval covered all choices, not only batch 1."
    ),
    "fresh_start": True,
    "choices": [
        {
            "id": number,
            "subject": subject,
            "status": "approved_as_proposed",
            "proposal_and_conditions": direction,
        }
        for number, subject, direction in approved_directions
    ],
}
assert len(approvals["choices"]) == 22

# Readback checks mounted Drive bytes; GPU recovery is not being tested here.
def save_verified_json(relative_path, value):
    path = TASK2_ROOT / relative_path
    payload = (
        json.dumps(value, indent=2, ensure_ascii=False) + "\n"
    ).encode("utf-8")
    temporary = path.with_name(path.name + ".pending")

    with temporary.open("xb") as stream:
        stream.write(payload)
        stream.flush()

    if temporary.read_bytes() != payload:
        raise IOError(f"Readback mismatch: {relative_path}")
    if path.exists():
        raise FileExistsError(f"Refusing to replace {path}")

    temporary.rename(path)
    if path.read_bytes() != payload:
        raise IOError(f"Final readback mismatch: {relative_path}")

    return {
        "path": str(path),
        "bytes": len(payload),
        "sha256": hashlib.sha256(payload).hexdigest(),
    }

# Inventory installed packages without importing/loading model libraries.
packages = {}
for name in (
    "torch", "transformers", "peft", "accelerate",
    "numpy", "Pillow", "huggingface-hub"
):
    try:
        packages[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        packages[name] = None

local_disk = shutil.disk_usage("/content")
report = {
    "timestamp": started.isoformat(),
    "run_id": run_id,
    "workspace_drive": str(TASK2_ROOT),
    "workspace_local": str(TASK2_LOCAL),
    "python": sys.version,
    "platform": platform.platform(),
    "packages_observed": packages,
    "gpu_detected": False,
    "models_loaded": False,
    "training_started": False,
    "dataset_downloads_started": False,
    "local_disk_free_GiB": round(local_disk.free / 2**30, 2),
    "drive_account_quota": (
        "NOT VERIFIED: mounted filesystem capacity is not account quota"
    ),
    "gpu_safety_status": (
        "No GPU task executed; training save/resume/release safeguards "
        "not yet tested"
    ),
}

saved_files = [
    save_verified_json("provenance/approved_choices.json", approvals),
    save_verified_json("logs/block_01_preflight.json", report),
]
save_verified_json(
    "provenance/block_01_saved_files.json",
    {"files": saved_files},
)

print("\nBLOCK 1 COMPLETE — CPU setup; all 22 approvals saved and read back.")
print(json.dumps(report, indent=2))
print("\nSaved records:")
for item in saved_files:
    print(item["path"], "\n  SHA256:", item["sha256"])

print("\nKeep hardware accelerator set to None.")
print("Paste this complete output into the chat.")

In [ ]:
# Block 2: CPU compatibility, revision pinning, and tokenizer/config files only.
import os, sys, json, uuid, shutil, hashlib, subprocess, traceback
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from importlib import metadata
from google.colab import runtime

ROOT = Path("/content/drive/MyDrive/ATML_PA3/task2/run_20261031_150949_1cff148a")
LOCAL = Path("/content/atml_pa3_task2/run_20261031_150949_1cff148a")
EXPECTED = {
    "provenance/approved_choices.json":
        "ec1e97439744d8321fedc8256e5c0f48e095518e4d17386271b3b2bda7965c33",
    "logs/block_01_preflight.json":
        "110ff8763f7aef0e2a1b2d75fcb4e55c11b1836babd3a4d265ca67ba312cbe0c",
}

# Keep preparation CPU-only.
accelerator = os.environ.get("COLAB_GPU", "0") not in ("", "0")
if shutil.which("nvidia-smi"):
    p = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    )
    accelerator |= p.returncode == 0 and bool(p.stdout.strip())

if accelerator or os.environ.get("COLAB_TPU_ADDR"):
    print("CPU task: requesting release of allocated accelerator.", flush=True)
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Reconnect with accelerator None and rerun.")

# Verify that we are continuing from your exact successful Block 1.
for relative, expected in EXPECTED.items():
    path = ROOT / relative
    if (
        not path.is_file()
        or hashlib.sha256(path.read_bytes()).hexdigest() != expected
    ):
        raise RuntimeError(
            f"Missing or changed Block 1 record: {relative}. "
            "Stop and share the output."
        )

LOCAL.mkdir(parents=True, exist_ok=True)
attempt = (
    datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y%m%d_%H%M%S_")
    + uuid.uuid4().hex[:8]
)

def save_record(relative, value):
    path = ROOT / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    data = (json.dumps(value, indent=2, ensure_ascii=False) + "\n").encode()

    if path.exists():
        if path.read_bytes() != data:
            raise RuntimeError(f"Refusing to change existing record: {relative}")
    else:
        temporary = path.with_name(
            path.name + "." + uuid.uuid4().hex + ".pending"
        )
        temporary.write_bytes(data)
        if temporary.read_bytes() != data:
            raise IOError(f"Readback failed: {relative}")
        temporary.rename(path)

    if path.read_bytes() != data:
        raise IOError(f"Final readback failed: {relative}")
    return hashlib.sha256(data).hexdigest()

report = {
    "attempt": attempt,
    "status": "started",
    "gpu_used": False,
    "weights_loaded": False,
    "image_downloads_started": False,
    "storage_evidence": {
        "source": "User screenshot, 31 October 2026",
        "Google_total_GB": 400,
        "Google_used_GB": 36.18,
        "Google_free_GB_approx": 363.82,
    },
    "packages_observed": {},
}

try:
    # Test existing packages; install or upgrade nothing.
    from huggingface_hub import HfApi, snapshot_download
    from transformers import AutoConfig, AutoTokenizer
    from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
    from transformers.models.siglip.modeling_siglip import SiglipVisionModel
    from peft import LoraConfig

    for name in (
        "torch", "transformers", "peft", "accelerate", "numpy",
        "Pillow", "huggingface-hub", "tokenizers"
    ):
        report["packages_observed"][name] = metadata.version(name)

    repos = {
        "qwen": "Qwen/Qwen3-0.6B",
        "siglip": "google/siglip-base-patch16-224",
    }
    lock_path = ROOT / "configs/model_revisions.json"
    api = HfApi(token=False)

    if lock_path.exists():
        pins = json.loads(lock_path.read_text())
        if set(pins) != set(repos):
            raise RuntimeError("Unexpected model revision lock.")
    else:
        pins = {}
        for name, repo in repos.items():
            info = api.model_info(repo, revision="main", timeout=30)
            pins[name] = {"repo_id": repo, "revision": info.sha}
        save_record("configs/model_revisions.json", pins)

    for name, repo in repos.items():
        if (
            pins[name]["repo_id"] != repo
            or len(pins[name]["revision"]) != 40
        ):
            raise RuntimeError("Invalid pinned model identity.")
    report["model_revisions"] = pins

    # Explicit whitelist excludes all model weight files.
    allowed = [
        "config.json", "generation_config.json", "tokenizer_config.json",
        "tokenizer.json", "tokenizer.model", "special_tokens_map.json",
        "vocab.json", "merges.txt", "added_tokens.json",
        "chat_template.jinja", "preprocessor_config.json",
    ]
    model_paths, inventory = {}, []

    for name, pin in pins.items():
        source = Path(snapshot_download(
            pin["repo_id"],
            revision=pin["revision"],
            allow_patterns=allowed,
            cache_dir=str(LOCAL / "hub"),
            token=False,
            max_workers=2,
        ))
        target = (
            ROOT / "provenance/model_assets" / name / pin["revision"]
        )
        target.mkdir(parents=True, exist_ok=True)

        for filename in allowed:
            src = source / filename
            if not src.is_file():
                continue

            data = src.read_bytes()
            dst = target / filename
            if dst.exists() and dst.read_bytes() != data:
                raise RuntimeError(f"Conflicting saved asset: {filename}")
            if not dst.exists():
                dst.write_bytes(data)
            if dst.read_bytes() != data:
                raise IOError(f"Asset readback failed: {filename}")

            inventory.append({
                "model": name,
                "filename": filename,
                "bytes": len(data),
                "sha256": hashlib.sha256(data).hexdigest(),
            })

        model_paths[name] = source

    qconfig = AutoConfig.from_pretrained(
        model_paths["qwen"],
        local_files_only=True,
        trust_remote_code=False,
    )
    sconfig = AutoConfig.from_pretrained(
        model_paths["siglip"],
        local_files_only=True,
        trust_remote_code=False,
    )
    vision = sconfig.vision_config

    assert qconfig.model_type == "qwen3" and qconfig.hidden_size == 1024
    assert sconfig.model_type == "siglip" and vision.hidden_size == 768
    assert vision.image_size == 224 and vision.patch_size == 16
    assert (vision.image_size // vision.patch_size) ** 2 == 196

    tokenizer = AutoTokenizer.from_pretrained(
        model_paths["qwen"],
        local_files_only=True,
        trust_remote_code=False,
    )

    # Diagnostic text only; this is not an experiment instruction.
    smoke = [{"role": "user", "content": "Interface smoke test."}]
    off = tokenizer.apply_chat_template(
        smoke, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    on = tokenizer.apply_chat_template(
        smoke, tokenize=False, add_generation_prompt=True,
        enable_thinking=True,
    )

    assert "enable_thinking" in tokenizer.get_chat_template()
    assert off != on, "Thinking switch did not change the serialized prompt."

    ids = tokenizer(off, add_special_tokens=False)["input_ids"]
    assert tokenizer.decode(ids, skip_special_tokens=False) == off
    assert tokenizer.eos_token_id == qconfig.eos_token_id
    assert tokenizer.encode(
        "<|im_end|>", add_special_tokens=False
    ) == [qconfig.eos_token_id]

    # Preserve native preprocessing settings without changing them.
    processor = json.loads(
        (model_paths["siglip"] / "preprocessor_config.json").read_text()
    )

    report.update({
        "status": "passed",
        "checks": {
            "model_classes_imported": True,
            "required_dimensions": True,
            "thinking_switch_effective": True,
            "tokenizer_roundtrip": True,
            "end_of_turn_token_verified": True,
        },
        "qwen_hidden_size": qconfig.hidden_size,
        "vision_hidden_size": vision.hidden_size,
        "visual_tokens": 196,
        "siglip_processor_settings": processor,
        "nonthinking_smoke_serialization": off,
        "smoke_test_is_experiment_prompt": False,
        "tokenizer_eos_id": tokenizer.eos_token_id,
        "tokenizer_pad_id": tokenizer.pad_token_id,
        "asset_file_count": len(inventory),
        "local_disk_free_GiB": round(
            shutil.disk_usage("/content").free / 2**30, 2
        ),
        "gpu_environment_validated": False,
    })
    save_record("provenance/model_assets_manifest.json", inventory)

except Exception:
    report["status"] = "failed"
    report["traceback"] = traceback.format_exc()
    failure_path = f"logs/block_02_{attempt}.json"
    save_record(failure_path, report)
    print(
        f"BLOCK 2 FAILED; diagnostics saved to {ROOT / failure_path}",
        flush=True,
    )
    raise

log_path = f"logs/block_02_{attempt}.json"
digest = save_record(log_path, report)

print("\nBLOCK 2 COMPLETE — CPU checks passed; no weights loaded.")
print(json.dumps(report, indent=2))
print("Saved log:", ROOT / log_path)
print("SHA256:", digest)
print("Keep accelerator None. Paste the complete output into the chat.")

In [ ]:
# Block 3: annotation downloads, integrity checks, and structure inventory.
# Continue in the SAME notebook after successful Block 2.
import gc, inspect, time, urllib.request, zipfile
from collections import Counter
from pathlib import Path

required_state = ("ROOT", "LOCAL", "save_record", "sconfig", "qconfig")
if any(name not in globals() for name in required_state):
    raise RuntimeError(
        "Block 2 state is missing. Stop and share this message; "
        "do not rerun Block 1."
    )

previous = ROOT / "logs/block_02_20261031_151449_9ecb9e8b.json"
assert hashlib.sha256(previous.read_bytes()).hexdigest() == (
    "e33edb522e01d7ef475949aa6e56692356ddbbf524769307abdc45fe0d4171aa"
)

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    print("CPU preparation: requesting runtime release.", flush=True)
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Select accelerator None before continuing.")

stamp = (
    datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y%m%d_%H%M%S_")
    + uuid.uuid4().hex[:8]
)
report3 = {
    "attempt": stamp,
    "status": "started",
    "gpu_used": False,
    "images_downloaded": False,
    "model_weights_loaded": False,
    "examples_selected": False,
    "archives": [],
    "json_inventory": [],
}

archives = LOCAL / "annotation_archives"
json_dir = LOCAL / "annotations"
archives.mkdir(parents=True, exist_ok=True)
json_dir.mkdir(parents=True, exist_ok=True)

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def copy_checked(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    wanted = file_sha(source)

    if target.exists():
        if file_sha(target) != wanted:
            raise IOError(f"Existing file differs; preserving it: {target}")
        return wanted

    temporary = target.with_name(
        target.name + "." + uuid.uuid4().hex + ".pending"
    )
    shutil.copyfile(source, temporary)
    if file_sha(temporary) != wanted:
        raise IOError(f"Copy readback failed: {target.name}")
    temporary.rename(target)
    return wanted

# COCO's documented public GCS bucket and current official VQAv2 links.
plan = [
    (
        "coco2014",
        "https://storage.googleapis.com/images.cocodataset.org/"
        "annotations/annotations_trainval2014.zip",
    ),
    (
        "vqa_train_annotations",
        "https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/"
        "v2_Annotations_Train_mscoco.zip",
    ),
    (
        "vqa_val_annotations",
        "https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/"
        "v2_Annotations_Val_mscoco.zip",
    ),
    (
        "vqa_train_questions",
        "https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/"
        "v2_Questions_Train_mscoco.zip",
    ),
    (
        "vqa_val_questions",
        "https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/"
        "v2_Questions_Val_mscoco.zip",
    ),
    (
        "vqa_train_pairs",
        "https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/"
        "v2_Complementary_Pairs_Train_mscoco.zip",
    ),
    (
        "vqa_val_pairs",
        "https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/"
        "v2_Complementary_Pairs_Val_mscoco.zip",
    ),
]
coco_needed = {
    "captions_train2014.json",
    "captions_val2014.json",
    "instances_train2014.json",
    "instances_val2014.json",
}

try:
    # Preserve warning provenance from the actual installed library.
    from transformers.models.siglip.configuration_siglip import SiglipTextConfig
    from transformers.models.siglip.modeling_siglip import SiglipVisionModel

    text_source = inspect.getsource(SiglipTextConfig)
    vision_source = inspect.getsource(SiglipVisionModel)
    text_cfg = sconfig.text_config

    warning_record = {
        "text_vocab_size": text_cfg.vocab_size,
        "text_bos_id": text_cfg.bos_token_id,
        "text_eos_id": text_cfg.eos_token_id,
        "qwen_eos_id_unchanged": qconfig.eos_token_id,
        "planned_encoder_class": "SiglipVisionModel",
        "siglip_text_branch_used_in_task2": False,
        "checkpoint_configuration_modified": False,
        "text_config_source_sha256":
            hashlib.sha256(text_source.encode()).hexdigest(),
        "vision_class_source_sha256":
            hashlib.sha256(vision_source.encode()).hexdigest(),
        "text_default_source_lines": [
            line.strip() for line in text_source.splitlines()
            if "bos_token_id" in line or "eos_token_id" in line
        ],
        "vision_normalization_source_lines": [
            line.strip() for line in vision_source.splitlines()
            if "post_layernorm" in line
        ],
        "vision_forward_numerically_tested": False,
    }
    assert warning_record["qwen_eos_id_unchanged"] == 151645
    save_record("provenance/siglip_warning_analysis.json", warning_record)
    report3["siglip_warning_analysis"] = warning_record

    for key, url in plan:
        local_zip = archives / f"{key}.zip"
        durable_zip = ROOT / "raw_annotation_archives" / f"{key}.zip"
        receipt_rel = f"manifests/annotation_downloads/{key}.json"
        receipt_path = ROOT / receipt_rel

        if durable_zip.exists():
            copy_checked(durable_zip, local_zip)

        if not local_zip.exists():
            partial = local_zip.with_suffix(".zip.part")
            offset = partial.stat().st_size if partial.exists() else 0
            headers = {"Range": f"bytes={offset}-"} if offset else {}
            print(
                f"Downloading {key} (resume offset {offset:,} bytes)...",
                flush=True,
            )

            request = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(request, timeout=60) as response:
                resumed = offset > 0 and response.status == 206
                if resumed and not response.headers.get(
                    "Content-Range", ""
                ).startswith(f"bytes {offset}-"):
                    raise IOError("Unexpected download resume position.")

                mode = "ab" if resumed else "wb"
                expected = response.headers.get("Content-Length")
                received, last_print = 0, time.monotonic()

                with partial.open(mode) as out:
                    while True:
                        chunk = response.read(4 * 1024 * 1024)
                        if not chunk:
                            break
                        out.write(chunk)
                        received += len(chunk)

                        if time.monotonic() - last_print > 20:
                            print(
                                f"  {partial.stat().st_size / 2**20:.1f}"
                                " MiB received",
                                flush=True,
                            )
                            last_print = time.monotonic()

                if expected is not None and received != int(expected):
                    raise IOError(
                        f"Incomplete download: {key}; partial retained."
                    )

            with zipfile.ZipFile(partial) as z:
                bad = z.testzip()
                if bad:
                    raise IOError(f"Archive CRC failed: {bad}")
            partial.rename(local_zip)

        digest = file_sha(local_zip)
        if receipt_path.exists():
            old = json.loads(receipt_path.read_text())
            if old["sha256"] != digest or old["url"] != url:
                raise IOError(f"Archive conflicts with saved receipt: {key}")

        with zipfile.ZipFile(local_zip) as z:
            bad = z.testzip()
            if bad:
                raise IOError(f"Archive CRC failed: {bad}")

            members = [
                m for m in z.infolist()
                if not m.is_dir()
                and m.filename.endswith(".json")
                and (
                    key != "coco2014"
                    or Path(m.filename).name in coco_needed
                )
            ]
            expected_count = 4 if key == "coco2014" else 1
            if len(members) != expected_count:
                raise RuntimeError(
                    f"Unexpected members in {key}: "
                    f"{[m.filename for m in members]}"
                )

            extracted = []
            for member in members:
                # Do not use archive-supplied directory paths.
                dest = json_dir / Path(member.filename).name
                temp = dest.with_name(dest.name + ".extracting")

                with z.open(member) as src, temp.open("wb") as out:
                    shutil.copyfileobj(src, out)

                if dest.exists():
                    if file_sha(dest) != file_sha(temp):
                        raise IOError(
                            f"Existing annotation differs: {dest.name}"
                        )
                    temp.unlink()
                else:
                    temp.rename(dest)

                extracted.append({
                    "filename": dest.name,
                    "bytes": dest.stat().st_size,
                    "sha256": file_sha(dest),
                })

        copy_checked(local_zip, durable_zip)
        receipt = {
            "key": key,
            "url": url,
            "bytes": local_zip.stat().st_size,
            "sha256": digest,
            "zip_crc_passed": True,
            "json_files": extracted,
            "hash_provenance": (
                "Observed download SHA256; "
                "not an official published checksum"
            ),
        }
        save_record(receipt_rel, receipt)
        report3["archives"].append(receipt)
        print(f"Verified and saved: {key}", flush=True)

    # Inspect one JSON at a time to limit CPU memory.
    # No example selection or split construction happens here.
    for path in sorted(json_dir.glob("*.json")):
        print("Inspecting", path.name, flush=True)
        with path.open() as f:
            data = json.load(f)

        row = {"filename": path.name, "bytes": path.stat().st_size}
        if isinstance(data, list):
            assert all(
                isinstance(pair, list)
                and len(pair) == 2
                and pair[0] != pair[1]
                for pair in data
            )
            row.update({
                "kind": "complementary_pairs",
                "pairs": len(data),
            })

        elif "images" in data:
            row.update({
                "kind": "coco_annotations",
                "images": len(data["images"]),
                "annotations": len(data["annotations"]),
            })
            if "instances_" in path.name:
                row["categories"] = len(data["categories"])
                row["crowd_annotations"] = sum(
                    bool(a.get("iscrowd", 0))
                    for a in data["annotations"]
                )

        elif "questions" in data:
            row.update({
                "kind": "vqa_questions",
                "questions": len(data["questions"]),
                "data_subtype": data.get("data_subtype"),
            })

        elif "annotations" in data:
            row.update({
                "kind": "vqa_answers",
                "questions": len(data["annotations"]),
                "answer_types": dict(Counter(
                    a["answer_type"] for a in data["annotations"]
                )),
                "all_examples_have_10_answers": all(
                    len(a["answers"]) == 10
                    for a in data["annotations"]
                ),
            })
            assert row["all_examples_have_10_answers"]

        else:
            raise RuntimeError(
                f"Unexpected JSON structure: {path.name}"
            )

        report3["json_inventory"].append(row)
        del data
        gc.collect()

    report3.update({
        "status": "passed",
        "local_annotation_directory": str(json_dir),
        "local_disk_free_GiB": round(
            shutil.disk_usage("/content").free / 2**30, 2
        ),
    })
    save_record(
        "manifests/annotation_inventory.json",
        report3["json_inventory"],
    )

except Exception:
    report3["status"] = "failed"
    report3["traceback"] = traceback.format_exc()
    save_record(f"logs/block_03_{stamp}.json", report3)
    print(
        "BLOCK 3 FAILED. Progress/diagnostics saved; "
        "keep accelerator None.",
        flush=True,
    )
    raise

log_name = f"logs/block_03_{stamp}.json"
digest = save_record(log_name, report3)

print(
    "\nBLOCK 3 COMPLETE — annotations saved and checked; "
    "no images or weights downloaded."
)
print(json.dumps({
    "status": report3["status"],
    "archives": len(report3["archives"]),
    "json_inventory": report3["json_inventory"],
    "siglip_warning_analysis": warning_record,
    "local_disk_free_GiB": report3["local_disk_free_GiB"],
}, indent=2))
print("Saved log:", ROOT / log_name)
print("SHA256:", digest)
print("Paste the complete output. Keep accelerator None.")

In [ ]:
# Repair 1: verify and save only the COCO annotation archive.
# Run in the SAME notebook. Do not rerun Blocks 1 or 2.
import urllib.request, zipfile, time, traceback

needed = ("ROOT", "LOCAL", "save_record", "file_sha", "copy_checked")
if any(name not in globals() for name in needed):
    raise RuntimeError("Notebook state changed. Stop and share this message.")
if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    raise RuntimeError("This repair needs accelerator None.")

EXPECTED_BYTES = 252872794
EXPECTED_SHA = (
    "031296bbc80c45a1d1f76bf9a90ead27e94e99ec629208449507a4917a3bf009"
)
CHECKSUM_SOURCE = (
    "https://raw.githubusercontent.com/tensorflow/datasets/master/"
    "tensorflow_datasets/url_checksums/coco_captions.txt"
)
CANDIDATES = [
    "https://s3.amazonaws.com/images.cocodataset.org/"
    "annotations/annotations_trainval2014.zip",
    "https://images.cocodataset.org/"
    "annotations/annotations_trainval2014.zip",
]

repair_id = (
    datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y%m%d_%H%M%S_")
    + uuid.uuid4().hex[:8]
)
repair = {
    "attempt": repair_id,
    "status": "started",
    "gpu_used": False,
    "original_failure": {
        "http_status": 404,
        "url": (
            "https://storage.googleapis.com/images.cocodataset.org/"
            "annotations/annotations_trainval2014.zip"
        ),
    },
    "checksum_source": CHECKSUM_SOURCE,
    "expected_bytes": EXPECTED_BYTES,
    "expected_sha256": EXPECTED_SHA,
    "probes": [],
    "downloaded_this_attempt": False,
}

local_zip = LOCAL / "annotation_archives/coco2014.zip"
durable_zip = ROOT / "raw_annotation_archives/coco2014.zip"
local_zip.parent.mkdir(parents=True, exist_ok=True)
source_lock = ROOT / "configs/coco2014_source.json"

try:
    if source_lock.exists():
        source_record = json.loads(source_lock.read_text())
        selected = source_record["url"]
        if selected not in CANDIDATES:
            raise RuntimeError("Unexpected saved COCO source.")
    else:
        selected = None
        for url in CANDIDATES:
            print("Checking HTTPS source:", url, flush=True)
            probe_record = {"url": url}
            try:
                request = urllib.request.Request(
                    url, headers={"Range": "bytes=0-3"}
                )
                with urllib.request.urlopen(request, timeout=20) as response:
                    magic = response.read(4)
                    probe_record.update({
                        "http_status": response.status,
                        "resolved_url": response.geturl(),
                        "zip_magic": magic.hex(),
                    })
                    if (
                        response.geturl().startswith("https://")
                        and magic == b"PK\x03\x04"
                    ):
                        selected = url

                repair["probes"].append(probe_record)
                if selected:
                    break
            except Exception as exc:
                probe_record["error"] = f"{type(exc).__name__}: {exc}"
                repair["probes"].append(probe_record)

        if selected is None:
            raise RuntimeError(
                "Neither HTTPS source returned a ZIP header; "
                "see saved diagnostics."
            )

    repair["selected_url"] = selected

    if durable_zip.exists():
        copy_checked(durable_zip, local_zip)

    if not local_zip.exists():
        partial = local_zip.with_name(
            f"coco2014_repair_{repair_id}.zip.part"
        )
        print(
            "Downloading COCO annotations; expected size "
            "252,872,794 bytes...",
            flush=True,
        )
        received, last_update = 0, time.monotonic()

        with urllib.request.urlopen(selected, timeout=60) as response:
            if not response.geturl().startswith("https://"):
                raise RuntimeError("Unexpected non-HTTPS redirect.")

            with partial.open("xb") as out:
                while True:
                    chunk = response.read(4 * 1024 * 1024)
                    if not chunk:
                        break
                    out.write(chunk)
                    received += len(chunk)

                    if received > EXPECTED_BYTES:
                        raise IOError("Response exceeded expected size.")

                    if time.monotonic() - last_update > 10:
                        print(
                            f"  {received / 2**20:.1f} / "
                            f"{EXPECTED_BYTES / 2**20:.1f} MiB",
                            flush=True,
                        )
                        last_update = time.monotonic()

        if (
            received != EXPECTED_BYTES
            or file_sha(partial) != EXPECTED_SHA
        ):
            raise IOError(
                "Size/checksum mismatch. Partial preserved; not accepted."
            )

        partial.rename(local_zip)
        repair["downloaded_this_attempt"] = True

    if (
        local_zip.stat().st_size != EXPECTED_BYTES
        or file_sha(local_zip) != EXPECTED_SHA
    ):
        raise IOError(
            "Cached archive does not match published checksum; preserving it."
        )

    with zipfile.ZipFile(local_zip) as z:
        bad = z.testzip()
        if bad:
            raise IOError(f"Archive CRC failed: {bad}")

        names = {Path(name).name for name in z.namelist()}
        required = {
            "captions_train2014.json",
            "captions_val2014.json",
            "instances_train2014.json",
            "instances_val2014.json",
        }
        if not required.issubset(names):
            raise IOError("Required COCO 2014 annotation files are missing.")

    copy_checked(local_zip, durable_zip)
    if file_sha(durable_zip) != EXPECTED_SHA:
        raise IOError("Drive-copy checksum mismatch.")

    source_record = {
        "url": selected,
        "archive_key": "coco2014",
        "expected_bytes": EXPECTED_BYTES,
        "expected_sha256": EXPECTED_SHA,
        "checksum_source": CHECKSUM_SOURCE,
    }
    save_record("configs/coco2014_source.json", source_record)

    repair.update({
        "status": "passed",
        "sha256": EXPECTED_SHA,
        "zip_crc_passed": True,
        "required_members_present": True,
        "drive_copy_readback_verified": True,
        "drive_archive": str(durable_zip),
        "remaining_annotation_archives_not_processed_by_this_repair": 6,
    })

except Exception:
    repair["status"] = "failed"
    repair["traceback"] = traceback.format_exc()
    save_record(f"logs/block_03_repair_{repair_id}.json", repair)
    print(json.dumps(repair, indent=2), flush=True)
    raise

log_path = f"logs/block_03_repair_{repair_id}.json"
digest = save_record(log_path, repair)

print(
    "\nCOCO REPAIR COMPLETE — published checksum, ZIP CRC, "
    "and Drive readback passed."
)
print(json.dumps(repair, indent=2))
print("Saved log:", ROOT / log_path)
print("Log SHA256:", digest)
print("Keep accelerator None. Paste the output.")
print("Do not rerun the original Block 3.")

In [ ]:
# Block 3B: finish annotation downloads and inventory after the COCO repair.
import os, json, uuid, hashlib, shutil, traceback, subprocess
import gc, time, urllib.request, zipfile
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter

needed = ("ROOT", "LOCAL", "save_record", "file_sha", "copy_checked")
if any(name not in globals() for name in needed):
    raise RuntimeError("Notebook state is missing. Stop; do not rerun Block 1.")

repair_log = ROOT / "logs/block_03_repair_20261031_152911_9193bda5.json"
assert file_sha(repair_log) == (
    "2bed1d7779467a12748a872cbb90cb47d875893c1265c16199ab1d9ea71f8104"
)
source = json.loads((ROOT / "configs/coco2014_source.json").read_text())
assert source["url"] == (
    "https://s3.amazonaws.com/images.cocodataset.org/"
    "annotations/annotations_trainval2014.zip"
)
assert source["expected_sha256"] == (
    "031296bbc80c45a1d1f76bf9a90ead27e94e99ec629208449507a4917a3bf009"
)

accelerator = os.environ.get("COLAB_GPU", "0") not in ("", "0")
if shutil.which("nvidia-smi"):
    probe = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    )
    accelerator |= probe.returncode == 0 and bool(probe.stdout.strip())

if accelerator or os.environ.get("COLAB_TPU_ADDR"):
    print("CPU preparation: requesting accelerator release.", flush=True)
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Reconnect with accelerator None.")

stamp = (
    datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y%m%d_%H%M%S_")
    + uuid.uuid4().hex[:8]
)
report3b = {
    "attempt": stamp,
    "status": "started",
    "gpu_used": False,
    "images_downloaded": False,
    "model_weights_loaded": False,
    "examples_selected": False,
    "archives": [],
    "json_inventory": [],
}

archives = LOCAL / "annotation_archives"
json_dir = LOCAL / "annotations"
archives.mkdir(parents=True, exist_ok=True)
json_dir.mkdir(parents=True, exist_ok=True)

base = "https://cvmlp.s3.amazonaws.com/vqa/mscoco/vqa/"
plan3b = [
    ("coco2014", source["url"]),
    ("vqa_train_annotations", base + "v2_Annotations_Train_mscoco.zip"),
    ("vqa_val_annotations", base + "v2_Annotations_Val_mscoco.zip"),
    ("vqa_train_questions", base + "v2_Questions_Train_mscoco.zip"),
    ("vqa_val_questions", base + "v2_Questions_Val_mscoco.zip"),
    ("vqa_train_pairs", base + "v2_Complementary_Pairs_Train_mscoco.zip"),
    ("vqa_val_pairs", base + "v2_Complementary_Pairs_Val_mscoco.zip"),
]
coco_needed = {
    "captions_train2014.json", "captions_val2014.json",
    "instances_train2014.json", "instances_val2014.json",
}

try:
    for key, url in plan3b:
        local_zip = archives / f"{key}.zip"
        durable_zip = ROOT / "raw_annotation_archives" / f"{key}.zip"
        receipt_rel = f"manifests/annotation_downloads/{key}.json"
        receipt_path = ROOT / receipt_rel

        if durable_zip.exists():
            copy_checked(durable_zip, local_zip)

        if not local_zip.exists():
            partial = local_zip.with_suffix(".zip.part")
            offset = partial.stat().st_size if partial.exists() else 0
            headers = {"Range": f"bytes={offset}-"} if offset else {}
            print(
                f"Downloading {key}; resume offset {offset:,} bytes...",
                flush=True,
            )

            request = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(request, timeout=60) as response:
                if not response.geturl().startswith("https://"):
                    raise RuntimeError("Unexpected non-HTTPS redirect.")

                resumed = offset > 0 and response.status == 206
                if resumed and not response.headers.get(
                    "Content-Range", ""
                ).startswith(f"bytes {offset}-"):
                    raise IOError("Unexpected download resume position.")

                received, last_update = 0, time.monotonic()
                expected = response.headers.get("Content-Length")

                with partial.open("ab" if resumed else "wb") as out:
                    while True:
                        chunk = response.read(4 * 1024 * 1024)
                        if not chunk:
                            break
                        out.write(chunk)
                        received += len(chunk)

                        if time.monotonic() - last_update > 15:
                            print(
                                f"  {partial.stat().st_size / 2**20:.1f} MiB",
                                flush=True,
                            )
                            last_update = time.monotonic()

                if expected is not None and received != int(expected):
                    raise IOError("Incomplete response; partial retained.")

            with zipfile.ZipFile(partial) as z:
                if z.testzip() is not None:
                    raise IOError(f"Archive CRC failed: {key}")
            partial.rename(local_zip)

        digest = file_sha(local_zip)
        if key == "coco2014":
            assert local_zip.stat().st_size == source["expected_bytes"]
            assert digest == source["expected_sha256"]

        if receipt_path.exists():
            old = json.loads(receipt_path.read_text())
            if old["url"] != url or old["sha256"] != digest:
                raise IOError(f"Archive differs from saved receipt: {key}")

        extracted = []
        with zipfile.ZipFile(local_zip) as z:
            if z.testzip() is not None:
                raise IOError(f"Archive CRC failed: {key}")

            members = [
                m for m in z.infolist()
                if not m.is_dir()
                and m.filename.endswith(".json")
                and (
                    key != "coco2014"
                    or Path(m.filename).name in coco_needed
                )
            ]
            if len(members) != (4 if key == "coco2014" else 1):
                raise RuntimeError(
                    f"Unexpected JSON members: {key}: "
                    f"{[m.filename for m in members]}"
                )

            for member in members:
                dest = json_dir / Path(member.filename).name
                temp = dest.with_name(dest.name + ".extracting")
                with z.open(member) as src, temp.open("wb") as out:
                    shutil.copyfileobj(src, out)

                if dest.exists():
                    if file_sha(dest) != file_sha(temp):
                        raise IOError(
                            f"Existing JSON differs; preserved: {dest.name}"
                        )
                    temp.unlink()
                else:
                    temp.rename(dest)

                extracted.append({
                    "filename": dest.name,
                    "bytes": dest.stat().st_size,
                    "sha256": file_sha(dest),
                })

        copy_checked(local_zip, durable_zip)
        receipt = {
            "key": key,
            "url": url,
            "bytes": local_zip.stat().st_size,
            "sha256": digest,
            "zip_crc_passed": True,
            "json_files": extracted,
            "hash_provenance": (
                "Published TensorFlow Datasets SHA256 verified"
                if key == "coco2014"
                else "Observed download SHA256; not a published reference checksum"
            ),
        }
        save_record(receipt_rel, receipt)
        report3b["archives"].append(receipt)
        print("Verified, extracted, and saved:", key, flush=True)

    # Parse one file at a time to limit CPU memory.
    files = sorted(json_dir.glob("*.json"))
    if len(files) != 10:
        raise RuntimeError(
            f"Expected 10 annotation JSON files, found {len(files)}."
        )

    for path in files:
        print("Inspecting", path.name, flush=True)
        with path.open() as f:
            data = json.load(f)

        split = "train2014" if "train2014" in path.name else "val2014"
        row = {
            "filename": path.name,
            "split": split,
            "bytes": path.stat().st_size,
        }

        if isinstance(data, list):
            assert all(
                isinstance(p, list) and len(p) == 2 and p[0] != p[1]
                for p in data
            )
            row.update({
                "kind": "complementary_pairs",
                "pairs": len(data),
            })

        elif "images" in data:
            row.update({
                "kind": "coco_annotations",
                "images": len(data["images"]),
                "annotations": len(data["annotations"]),
            })
            assert row["images"] == (
                82783 if split == "train2014" else 40504
            )
            if "instances_" in path.name:
                row["categories"] = len(data["categories"])
                row["crowd_annotations"] = sum(
                    bool(a.get("iscrowd", 0))
                    for a in data["annotations"]
                )
                assert row["categories"] == 80

        elif "questions" in data:
            row.update({
                "kind": "vqa_questions",
                "questions": len(data["questions"]),
                "data_subtype": data.get("data_subtype"),
            })
            assert row["questions"] == (
                443757 if split == "train2014" else 214354
            )
            assert row["data_subtype"] == split

        elif "annotations" in data:
            row.update({
                "kind": "vqa_answers",
                "questions": len(data["annotations"]),
                "answer_types": dict(Counter(
                    a["answer_type"] for a in data["annotations"]
                )),
                "all_examples_have_10_answers": all(
                    len(a["answers"]) == 10
                    for a in data["annotations"]
                ),
            })
            assert row["questions"] == (
                443757 if split == "train2014" else 214354
            )
            assert row["all_examples_have_10_answers"]

        else:
            raise RuntimeError(f"Unexpected JSON structure: {path.name}")

        report3b["json_inventory"].append(row)
        del data
        gc.collect()

    save_record(
        "manifests/annotation_inventory.json",
        report3b["json_inventory"],
    )
    report3b.update({
        "status": "passed",
        "local_annotation_directory": str(json_dir),
        "local_disk_free_GiB": round(
            shutil.disk_usage("/content").free / 2**30, 2
        ),
    })

except Exception:
    report3b["status"] = "failed"
    report3b["traceback"] = traceback.format_exc()
    save_record(f"logs/block_03b_{stamp}.json", report3b)
    print(
        "BLOCK 3B FAILED. Verified archives and diagnostics retained; "
        "no GPU used.",
        flush=True,
    )
    raise

log_name = f"logs/block_03b_{stamp}.json"
digest = save_record(log_name, report3b)

print("\nBLOCK 3B COMPLETE — 7 archives and 10 annotation JSON files checked.")
print(json.dumps({
    "status": report3b["status"],
    "json_inventory": report3b["json_inventory"],
    "local_disk_free_GiB": report3b["local_disk_free_GiB"],
    "images_downloaded": False,
    "model_weights_loaded": False,
    "examples_selected": False,
}, indent=2))
print("Saved log:", ROOT / log_name)
print("SHA256:", digest)
print("Keep accelerator None. Paste the complete output.")

In [ ]:
# Block 4A: answer normalization and natural-pair eligibility. CPU only.
import os, re, ast, json, uuid, hashlib, shutil, traceback
import urllib.request, warnings, gc
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter
from types import SimpleNamespace
from functools import lru_cache

required = (
    "ROOT", "LOCAL", "save_record", "file_sha", "copy_checked", "tokenizer"
)
if any(name not in globals() for name in required):
    raise RuntimeError(
        "Required notebook state missing. Stop; do not restart Block 1."
    )

previous = ROOT / "logs/block_03b_20261031_153258_5997ff44.json"
assert file_sha(previous) == (
    "c43bd46506db8a534fe478e6f72fb385bbc6b2370c1efec5f4b48164f3fbdf5d"
)

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    print("CPU task: requesting release of allocated GPU.", flush=True)
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Reconnect with accelerator None.")

stamp = (
    datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y%m%d_%H%M%S_")
    + uuid.uuid4().hex[:8]
)
report4 = {
    "attempt": stamp,
    "status": "started",
    "gpu_used": False,
    "experiment_subsets_selected": False,
    "final_test_frozen": False,
    "splits": {},
}
annotation_dir = LOCAL / "annotations"
work4 = LOCAL / "eligibility"
work4.mkdir(parents=True, exist_ok=True)

try:
    # Verify raw JSON files against their saved download receipts.
    for receipt_path in sorted(
        (ROOT / "manifests/annotation_downloads").glob("*.json")
    ):
        receipt = json.loads(receipt_path.read_text())
        for item in receipt["json_files"]:
            path = annotation_dir / item["filename"]
            if (
                not path.is_file()
                or file_sha(path) != item["sha256"]
            ):
                raise IOError(
                    f"Annotation missing or changed: {item['filename']}"
                )

    # Snapshot the official source once; reuse those exact bytes afterward.
    upstream_url = (
        "https://raw.githubusercontent.com/GT-Vision-Lab/VQA/master/"
        "PythonEvaluationTools/vqaEvaluation/vqaEval.py"
    )
    snapshot = ROOT / "provenance/external/vqaEval_upstream.py"
    source_receipt = ROOT / "provenance/vqa_source_snapshot.json"

    if source_receipt.exists():
        info = json.loads(source_receipt.read_text())
        assert info["url"] == upstream_url
        assert file_sha(snapshot) == info["sha256"]
        raw = snapshot.read_bytes()
    else:
        print("Retrieving official VQA normalization source...", flush=True)
        with urllib.request.urlopen(upstream_url, timeout=30) as response:
            raw = response.read()
        local_source = work4 / "vqaEval_upstream.py"
        local_source.write_bytes(raw)
        copy_checked(local_source, snapshot)
        info = {
            "url": upstream_url,
            "sha256": hashlib.sha256(raw).hexdigest(),
            "attribution": (
                "GT-Vision-Lab/VQA; original author aagrawal; "
                "original source retained"
            ),
        }
        save_record("provenance/vqa_source_snapshot.json", info)

    upstream = raw.decode("utf-8")

    # Reuse only three reviewed methods compatible with Python 3.
    # The old evaluate() routine is not executed.
    method_names = (
        "__init__", "processPunctuation", "processDigitArticle"
    )

    def extract_method(name):
        match = re.search(
            r"(?m)^([\t ]+)def " + re.escape(name) + r"\(",
            upstream,
        )
        if not match:
            raise RuntimeError(f"Official method missing: {name}")
        next_method = re.search(
            r"(?m)^" + re.escape(match.group(1)) + r"def \w+\(",
            upstream[match.end():],
        )
        end = (
            match.end() + next_method.start()
            if next_method else len(upstream)
        )
        return upstream[match.start():end].rstrip() + "\n\n"

    selected_class = (
        "class VQAEval:\n"
        + "".join(extract_method(n) for n in method_names)
    )

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", SyntaxWarning)
        tree = ast.parse(selected_class)

    assert len(tree.body) == 1
    assert isinstance(tree.body[0], ast.ClassDef)
    assert [n.name for n in tree.body[0].body] == list(method_names)

    allowed_calls = {
        "getQuesIds", "compile", "search", "replace", "sub", "lower",
        "split", "setdefault", "append", "join",
    }
    for node in ast.walk(tree):
        if isinstance(
            node, (ast.Import, ast.ImportFrom, ast.Global, ast.Nonlocal)
        ):
            raise RuntimeError(
                "Unexpected construct in selected upstream methods."
            )
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                assert node.func.id == "enumerate"
            else:
                assert isinstance(node.func, ast.Attribute)
                assert node.func.attr in allowed_calls

    namespace = {"re": re, "__name__": "task2_vqa_normalization"}
    exec(
        compile(tree, "<selected official VQA methods>", "exec"),
        namespace,
    )
    official = namespace["VQAEval"](
        SimpleNamespace(getQuesIds=lambda: []), None
    )

    @lru_cache(maxsize=200000)
    def normalize_answer(text):
        text = text.replace("\n", " ").replace("\t", " ").strip()
        return official.processDigitArticle(
            official.processPunctuation(text)
        )

    tests = {
        "The two cats.": "2 cats",
        "1,000": "1000",
        "3.14": "3.14",
        "dont": "don't",
        " YES!\n": "yes",
        "": "",
    }
    for text, expected in tests.items():
        assert normalize_answer(text) == expected, (
            text, normalize_answer(text), expected
        )

    # Save a reusable module so training/evaluation use the same normalization.
    wrapper = '''
from types import SimpleNamespace
from functools import lru_cache
_normalizer = VQAEval(SimpleNamespace(getQuesIds=lambda: []), None)
@lru_cache(maxsize=200000)
def normalize_answer(text):
    text = text.replace("\\n", " ").replace("\\t", " ").strip()
    return _normalizer.processDigitArticle(_normalizer.processPunctuation(text))
'''
    module_text = (
        "# Normalization methods reused from GT-Vision-Lab/VQA (aagrawal).\n"
        "# Full upstream source and SHA256 are saved under provenance/.\n"
        "import re\n\n"
        + selected_class + wrapper
    )
    module_path = work4 / "vqa_normalization.py"
    module_path.write_text(module_text)
    copy_checked(module_path, ROOT / "code/vqa_normalization.py")

    normalizer_info = {
        "upstream": info,
        "module_sha256": file_sha(module_path),
        "methods_reused": list(method_names),
        "normalization_smoke_tests": len(tests),
        "normalization_policy": (
            "Always apply the official punctuation and digit/article "
            "helpers to each answer"
        ),
        "legacy_evaluate_not_used": True,
        "syntax_warnings": [str(w.message) for w in caught],
    }
    save_record("provenance/vqa_normalizer.json", normalizer_info)
    report4["normalizer"] = normalizer_info

    @lru_cache(maxsize=100000)
    def content_length(text):
        return len(tokenizer.encode(text, add_special_tokens=False))

    for split in ("train2014", "val2014"):
        print(f"\nPreparing {split} candidate pairs...", flush=True)

        with (
            annotation_dir / f"v2_mscoco_{split}_complementary_pairs.json"
        ).open() as f:
            raw_pairs = json.load(f)

        pairs = sorted(tuple(sorted(map(int, p))) for p in raw_pairs)
        needed_ids = {qid for pair in pairs for qid in pair}
        del raw_pairs

        with (
            annotation_dir
            / f"v2_OpenEnded_mscoco_{split}_questions.json"
        ).open() as f:
            payload = json.load(f)

        questions = {
            int(q["question_id"]): (int(q["image_id"]), q["question"])
            for q in payload["questions"]
            if int(q["question_id"]) in needed_ids
        }
        del payload
        assert set(questions) == needed_ids

        with (
            annotation_dir / f"v2_mscoco_{split}_annotations.json"
        ).open() as f:
            payload = json.load(f)

        answers = {}
        for index, annotation in enumerate(payload["annotations"]):
            qid = int(annotation["question_id"])
            if qid not in needed_ids:
                continue

            assert qid not in answers
            assert int(annotation["image_id"]) == questions[qid][0]

            refs = sorted(
                annotation["answers"],
                key=lambda a: int(a["answer_id"]),
            )
            assert len(refs) == 10
            assert len({int(a["answer_id"]) for a in refs}) == 10

            normalized = [
                normalize_answer(a["answer"]) for a in refs
            ]
            canonical, agreement = Counter(normalized).most_common(1)[0]
            target_ref = next(
                a for a, norm in zip(refs, normalized)
                if norm == canonical
            )
            answers[qid] = {
                "answer_type": annotation["answer_type"],
                "canonical": canonical,
                "agreement": agreement,
                "target": target_ref["answer"],
                "target_answer_id": int(target_ref["answer_id"]),
                "human_answers": [a["answer"] for a in refs],
                "answer_ids": [int(a["answer_id"]) for a in refs],
            }

            if (index + 1) % 50000 == 0:
                print(
                    f"  normalized {index + 1:,} source questions",
                    flush=True,
                )

        del payload
        gc.collect()
        assert set(answers) == needed_ids

        counts, rejected_counts, flags = Counter(), Counter(), Counter()
        seen = set()
        eligible_path = work4 / f"natural_eligible_{split}.jsonl"
        rejected_path = work4 / f"natural_rejected_{split}.jsonl"
        eligible_temp = eligible_path.with_suffix(".jsonl.pending")
        rejected_temp = rejected_path.with_suffix(".jsonl.pending")

        with eligible_temp.open("w") as good, rejected_temp.open("w") as bad:
            for pair in pairs:
                q1, q2 = pair
                a, b = answers[q1], answers[q2]
                im1, question1 = questions[q1]
                im2, question2 = questions[q2]

                reason = None
                if pair in seen:
                    reason = "duplicate_unordered_question_pair"
                elif im1 == im2:
                    reason = "same_image"
                elif question1 != question2:
                    reason = "question_strings_differ"
                elif min(a["agreement"], b["agreement"]) < 8:
                    reason = "consensus_below_8_of_10"
                elif a["canonical"] == b["canonical"]:
                    reason = "normalized_consensus_answers_equal"
                elif a["answer_type"] != b["answer_type"]:
                    reason = "mixed_answer_types_unassigned_to_quota"
                elif content_length(question1) > 80:
                    reason = "question_content_above_80_tokens"
                elif max(
                    content_length(a["target"]),
                    content_length(b["target"]),
                ) > 16:
                    reason = "chosen_original_target_above_16_tokens"

                seen.add(pair)
                if reason:
                    rejected_counts[reason] += 1
                    bad.write(json.dumps({
                        "question_ids": pair,
                        "reason": reason,
                    }, ensure_ascii=False) + "\n")
                    continue

                assert a["answer_type"] in ("yes/no", "number", "other")
                examples = []
                for qid, image_id, answer in (
                    (q1, im1, a), (q2, im2, b)
                ):
                    examples.append(dict(
                        question_id=qid,
                        image_id=image_id,
                        **answer,
                        target_content_tokens=content_length(
                            answer["target"]
                        ),
                    ))

                row = {
                    "pair_id": f"{split}:{q1}:{q2}",
                    "source_split": split,
                    "answer_type": a["answer_type"],
                    "question": question1,
                    "question_content_tokens": content_length(question1),
                    "examples": examples,
                }

                # Retain and flag these cases for review; do not silently drop.
                if not question1.strip():
                    flags["empty_question_pairs"] += 1
                if not a["canonical"] or not b["canonical"]:
                    flags["empty_normalized_canonical_pairs"] += 1
                if not a["target"].strip() or not b["target"].strip():
                    flags["empty_original_target_pairs"] += 1

                counts[a["answer_type"]] += 1
                good.write(json.dumps(row, ensure_ascii=False) + "\n")

        for temp, final in (
            (eligible_temp, eligible_path),
            (rejected_temp, rejected_path),
        ):
            if final.exists():
                if file_sha(temp) != file_sha(final):
                    raise IOError(
                        f"Existing eligibility output differs: {final.name}"
                    )
                temp.unlink()
            else:
                temp.rename(final)

        outputs = []
        for path in (eligible_path, rejected_path):
            destination = ROOT / "manifests/eligibility" / path.name
            digest = copy_checked(path, destination)
            outputs.append({
                "path": str(destination),
                "sha256": digest,
                "bytes": path.stat().st_size,
            })

        minimums = (
            {"yes/no": 640, "number": 256, "other": 640}
            if split == "train2014"
            else {"yes/no": 48, "number": 32, "other": 48}
        )
        summary = {
            "supplied_pairs": len(pairs),
            "eligible_by_answer_type": dict(counts),
            "rejected_first_reason_counts": dict(rejected_counts),
            "review_flags": dict(flags),
            "minimum_pair_counts_before_image_disjoint_allocation": minimums,
            "basic_count_feasible": all(
                counts[t] >= n for t, n in minimums.items()
            ),
            "image_disjoint_allocation_tested": False,
            "outputs": outputs,
        }

        assert (
            sum(counts.values()) + sum(rejected_counts.values())
            == len(pairs)
        )
        report4["splits"][split] = summary
        save_record(
            f"manifests/eligibility/natural_{split}_summary.json",
            summary,
        )
        print(json.dumps({split: summary}, indent=2), flush=True)

        del questions, answers, pairs, needed_ids, seen
        gc.collect()

    report4["status"] = "passed"
    report4["local_disk_free_GiB"] = round(
        shutil.disk_usage("/content").free / 2**30, 2
    )

except Exception:
    report4["status"] = "failed"
    report4["traceback"] = traceback.format_exc()
    save_record(f"logs/block_04a_{stamp}.json", report4)
    print(
        "BLOCK 4A FAILED; diagnostics saved. Keep accelerator None.",
        flush=True,
    )
    raise

log_path = f"logs/block_04a_{stamp}.json"
digest = save_record(log_path, report4)
print(
    "\nBLOCK 4A COMPLETE — normalization and eligibility analysis; "
    "no experiment subsets selected."
)
print(json.dumps(report4, indent=2))
print("Saved log:", ROOT / log_path)
print("SHA256:", digest)
print("Keep accelerator None. Paste the complete output.")

In [ ]:
# Block 4B: inspect answer edge cases and caption eligibility. CPU only.
import os, json, uuid, shutil, traceback, gc, warnings, importlib.util
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter, defaultdict

needed = (
    "ROOT", "LOCAL", "save_record", "file_sha", "copy_checked", "tokenizer"
)
if any(n not in globals() for n in needed):
    raise RuntimeError("Notebook state missing. Stop; do not rerun Block 1.")

previous = ROOT / "logs/block_04a_20261031_154343_2e9b562d.json"
assert file_sha(previous) == (
    "78d19d8f125aade489c94c1ca23fb82212d4168153cc40f09af3193be1bacd1e"
)

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    print("CPU task: requesting GPU release.", flush=True)
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Reconnect with accelerator None.")

stamp = (
    datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y%m%d_%H%M%S_")
    + uuid.uuid4().hex[:8]
)
report4b = {
    "attempt": stamp,
    "status": "started",
    "gpu_used": False,
    "subsets_selected": False,
    "new_exclusions_applied": False,
    "natural_audits": {},
    "caption_eligibility": {},
}
annotations = LOCAL / "annotations"
scratch = LOCAL / "eligibility"
scratch.mkdir(parents=True, exist_ok=True)

try:
    # Import the exact normalization module verified in Block 4A.
    module_path = ROOT / "code/vqa_normalization.py"
    assert file_sha(module_path) == (
        "56251cdcddbad50cc5e31cb83744cf1752267390b7d4e315dfe00e9edf905d0d"
    )
    spec = importlib.util.spec_from_file_location(
        "task2_checked_vqa_normalizer", module_path
    )
    module = importlib.util.module_from_spec(spec)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", SyntaxWarning)
        spec.loader.exec_module(module)
    norm = module.normalize_answer
    report4b["normalizer_import_warnings"] = [
        str(w.message) for w in caught
    ]

    for split in ("train2014", "val2014"):
        summary = json.loads((
            ROOT / f"manifests/eligibility/natural_{split}_summary.json"
        ).read_text())
        paths = {}

        for item in summary["outputs"]:
            durable = Path(item["path"])
            local = scratch / durable.name
            copy_checked(durable, local)
            assert file_sha(local) == item["sha256"]
            kind = "rejected" if "rejected_" in local.name else "eligible"
            paths[kind] = local

        flagged = []
        with paths["eligible"].open() as stream:
            for line in stream:
                row = json.loads(line)
                if any(not ex["canonical"] for ex in row["examples"]):
                    for ex in row["examples"]:
                        assert norm(ex["target"]) == ex["canonical"]
                        assert sum(
                            norm(a) == ex["canonical"]
                            for a in ex["human_answers"]
                        ) == ex["agreement"]
                    flagged.append(row)

        assert len(flagged) == summary["review_flags"].get(
            "empty_normalized_canonical_pairs", 0
        )
        empty_targets = Counter(
            ex["target"]
            for row in flagged
            for ex in row["examples"]
            if not ex["canonical"]
        )

        # Inspect every pair currently unassigned because answer types differ.
        mixed_ids = []
        with paths["rejected"].open() as stream:
            for line in stream:
                row = json.loads(line)
                if row["reason"] == "mixed_answer_types_unassigned_to_quota":
                    mixed_ids.append(tuple(row["question_ids"]))

        ids = {qid for pair in mixed_ids for qid in pair}

        with (
            annotations / f"v2_OpenEnded_mscoco_{split}_questions.json"
        ).open() as stream:
            payload = json.load(stream)
        qmap = {
            int(q["question_id"]): q
            for q in payload["questions"]
            if int(q["question_id"]) in ids
        }
        del payload

        with (
            annotations / f"v2_mscoco_{split}_annotations.json"
        ).open() as stream:
            payload = json.load(stream)
        amap = {}
        for annotation in payload["annotations"]:
            qid = int(annotation["question_id"])
            if qid in ids:
                refs = sorted(
                    annotation["answers"],
                    key=lambda a: int(a["answer_id"]),
                )
                canonical, agreement = Counter(
                    norm(a["answer"]) for a in refs
                ).most_common(1)[0]
                amap[qid] = {
                    "question_id": qid,
                    "image_id": int(annotation["image_id"]),
                    "question": qmap[qid]["question"],
                    "answer_type": annotation["answer_type"],
                    "canonical": canonical,
                    "agreement": agreement,
                    "human_answers": [a["answer"] for a in refs],
                }

        del payload
        gc.collect()
        assert set(amap) == ids

        mixed_cases, mixed_types = [], Counter()
        for pair in sorted(mixed_ids):
            examples = [amap[qid] for qid in pair]
            key = " | ".join(sorted(
                ex["answer_type"] for ex in examples
            ))
            mixed_types[key] += 1
            mixed_cases.append({
                "question_ids": pair,
                "examples": examples,
            })

        audit = {
            "source_split": split,
            "empty_normalization_pairs": flagged,
            "empty_normalization_original_targets": dict(empty_targets),
            "mixed_answer_type_pairs": mixed_cases,
            "mixed_type_counts": dict(mixed_types),
            "no_new_exclusion_policy_applied": True,
        }
        audit_rel = (
            f"manifests/eligibility/natural_edge_case_audit_{split}.json"
        )
        audit_hash = save_record(audit_rel, audit)

        display_cases = [
            {
                "pair_id": row["pair_id"],
                "question": row["question"],
                "answer_type": row["answer_type"],
                "examples": [
                    {
                        "image_id": ex["image_id"],
                        "target": ex["target"],
                        "canonical": ex["canonical"],
                        "agreement": ex["agreement"],
                        "raw_answer_counts": dict(Counter(
                            ex["human_answers"]
                        )),
                    }
                    for ex in row["examples"]
                ],
            }
            for row in flagged
        ]
        report4b["natural_audits"][split] = {
            "empty_normalization_pair_count": len(flagged),
            "empty_normalization_original_targets": dict(empty_targets),
            "empty_normalization_cases": display_cases,
            "mixed_answer_type_pair_count": len(mixed_cases),
            "mixed_type_counts": dict(mixed_types),
            "mixed_diagnostic_examples_first_3_by_pair_ids": mixed_cases[:3],
            "audit_path": str(ROOT / audit_rel),
            "audit_sha256": audit_hash,
        }

        del flagged, mixed_cases, qmap, amap, ids
        gc.collect()

    # Check original caption content against the prescribed 64-token cap.
    coco_receipt = json.loads((
        ROOT / "manifests/annotation_downloads/coco2014.json"
    ).read_text())
    caption_hashes = {
        r["filename"]: r["sha256"]
        for r in coco_receipt["json_files"]
    }
    assert tokenizer.eos_token_id == 151645

    for split in ("train2014", "val2014"):
        print(f"\nChecking {split} captions...", flush=True)
        raw_path = annotations / f"captions_{split}.json"
        assert file_sha(raw_path) == caption_hashes[raw_path.name]

        with raw_path.open() as stream:
            payload = json.load(stream)
        image_ids = {int(im["id"]) for im in payload["images"]}
        records = sorted(
            payload["annotations"], key=lambda a: int(a["id"])
        )
        assert len({int(a["id"]) for a in records}) == len(records)

        per_image = Counter()
        distinct_text = defaultdict(set)
        totals = Counter()
        local_output = scratch / f"caption_candidates_{split}.jsonl"
        temp = local_output.with_suffix(".jsonl.pending")

        with temp.open("w") as stream:
            for start in range(0, len(records), 512):
                batch = records[start:start + 512]
                encoded = tokenizer(
                    [a["caption"] for a in batch],
                    add_special_tokens=False,
                    padding=False,
                    truncation=False,
                    return_attention_mask=False,
                )["input_ids"]
                assert len(encoded) == len(batch)

                for annotation, tokens in zip(batch, encoded):
                    image_id = int(annotation["image_id"])
                    caption_id = int(annotation["id"])
                    assert image_id in image_ids

                    text, length = annotation["caption"], len(tokens)
                    within_cap = length <= 64
                    empty = not text.strip()

                    totals["all_captions"] += 1
                    totals["within_cap" if within_cap else "above_cap"] += 1
                    totals["empty_or_whitespace"] += int(empty)

                    if within_cap:
                        per_image[image_id] += 1
                        totals["duplicate_eligible_caption_texts"] += int(
                            text in distinct_text[image_id]
                        )
                        distinct_text[image_id].add(text)

                    stream.write(json.dumps({
                        "caption_id": caption_id,
                        "image_id": image_id,
                        "caption": text,
                        "content_tokens": length,
                        "within_content_cap": within_cap,
                        "empty_or_whitespace": empty,
                    }, ensure_ascii=False) + "\n")

                if start and start % (512 * 100) == 0:
                    print(
                        f"  checked {min(start + 512, len(records)):,} captions",
                        flush=True,
                    )

        if local_output.exists():
            if file_sha(local_output) != file_sha(temp):
                raise IOError(
                    "Caption index differs from previous output; preserving it."
                )
            temp.unlink()
        else:
            temp.rename(local_output)

        target = ROOT / "manifests/eligibility" / local_output.name
        digest = copy_checked(local_output, target)
        needed_per_image = 3 if split == "train2014" else 1
        qualifying = sum(
            len(distinct_text[image_id]) >= needed_per_image
            for image_id in image_ids
        )
        caption_summary = {
            "source_split": split,
            "images": len(image_ids),
            "counts": dict(totals),
            "distinct_caption_texts_required_per_image": needed_per_image,
            "images_with_enough_distinct_caption_texts_within_64_tokens":
                qualifying,
            "required_image_count_before_evaluation_image_exclusions":
                6000 if split == "train2014" else 96,
            "image_disjoint_allocation_tested": False,
            "subsets_selected": False,
            "index_path": str(target),
            "index_sha256": digest,
            "index_bytes": target.stat().st_size,
        }
        report4b["caption_eligibility"][split] = caption_summary
        save_record(
            f"manifests/eligibility/caption_{split}_summary.json",
            caption_summary,
        )
        print(json.dumps(caption_summary, indent=2), flush=True)

        del payload, records, image_ids, per_image, distinct_text
        gc.collect()

    report4b.update({
        "status": "passed",
        "local_disk_free_GiB": round(
            shutil.disk_usage("/content").free / 2**30, 2
        ),
    })

except Exception:
    report4b["status"] = "failed"
    report4b["traceback"] = traceback.format_exc()
    save_record(f"logs/block_04b_{stamp}.json", report4b)
    print(
        "BLOCK 4B FAILED; diagnostics saved. Keep accelerator None.",
        flush=True,
    )
    raise

log_path = f"logs/block_04b_{stamp}.json"
digest = save_record(log_path, report4b)
print(
    "\nBLOCK 4B COMPLETE — edge-case audit and caption counts; "
    "no subsets selected."
)
print(json.dumps(report4b, indent=2))
print("Saved log:", ROOT / log_path)
print("SHA256:", digest)
print("Keep accelerator None. Paste the complete output.")

In [ ]:
# Block 5A: approved exclusions, natural-VQA subsets, and training captions.
# CPU only. Grounding evaluation and caption development come later.
import os, json, uuid, shutil, traceback, gc
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter, defaultdict
import numpy as np

required = ("ROOT", "LOCAL", "save_record", "file_sha", "copy_checked")
if any(n not in globals() for n in required):
    raise RuntimeError("Notebook state missing. Stop; do not rerun Block 1.")

prior_path = ROOT / "logs/block_04b_20261031_155026_82d3640b.json"
assert file_sha(prior_path) == (
    "35c8314906b0f5cdab93f998159f43b493fc2e111cc058cea957b2053dd23945"
)
prior = json.loads(prior_path.read_text())

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    print("CPU selection: requesting GPU release.", flush=True)
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Reconnect with accelerator None.")

stamp = (
    datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y%m%d_%H%M%S_")
    + uuid.uuid4().hex[:8]
)
report5 = {
    "attempt": stamp,
    "status": "started",
    "gpu_used": False,
    "complete_dataset_frozen": False,
    "allocation": {},
    "outputs": [],
}
local5 = LOCAL / "selection"
local5.mkdir(parents=True, exist_ok=True)

def choose_pairs(rows, quotas, rng, blocked_images=()):
    blocked_images = set(blocked_images)
    selected = []
    for answer_type in ("yes/no", "number", "other"):
        candidates = sorted(
            (r for r in rows if r["answer_type"] == answer_type),
            key=lambda r: tuple(
                int(ex["question_id"]) for ex in r["examples"]
            ),
        )
        count = 0
        for index in rng.permutation(len(candidates)):
            row = candidates[int(index)]
            images = {int(ex["image_id"]) for ex in row["examples"]}
            if images & blocked_images:
                continue
            selected.append(row)
            count += 1
            if count == quotas[answer_type]:
                break
        if count != quotas[answer_type]:
            raise RuntimeError(
                f"Insufficient {answer_type} pairs after image exclusions: "
                f"{count}/{quotas[answer_type]}"
            )
    return selected

def pair_images(rows):
    return {
        int(ex["image_id"])
        for row in rows
        for ex in row["examples"]
    }

def save_rows(filename, rows):
    path = local5 / filename
    temp = path.with_name(path.name + ".pending")
    with temp.open("w") as out:
        for row in rows:
            out.write(json.dumps(row, ensure_ascii=False) + "\n")
    if path.exists():
        if file_sha(path) != file_sha(temp):
            raise IOError(
                f"Existing selection differs; preserving it: {filename}"
            )
        temp.unlink()
    else:
        temp.rename(path)

    durable = ROOT / "manifests/selected" / filename
    digest = copy_checked(path, durable)
    return {
        "path": str(durable),
        "sha256": digest,
        "rows": len(rows),
        "bytes": path.stat().st_size,
    }

try:
    policy = {
        "decision_id": "A004",
        "date": "2026-10-31",
        "user_response": "Approve excluding both groups (recommended)",
        "exclude_empty_normalized_canonical_pairs": True,
        "exclude_mixed_official_answer_type_pairs": True,
        "normalization_changed": False,
        "basis": (
            "Prediction-independent reference/type audit before subset selection"
        ),
    }
    save_record("configs/additional_eligibility_A004.json", policy)

    # Preserve every newly excluded pair ID.
    exclusions = []
    for split in ("train2014", "val2014"):
        audit_info = prior["natural_audits"][split]
        audit_path = Path(audit_info["audit_path"])
        assert file_sha(audit_path) == audit_info["audit_sha256"]
        audit = json.loads(audit_path.read_text())

        for row in audit["empty_normalization_pairs"]:
            exclusions.append({
                "pair_id": row["pair_id"],
                "source_split": split,
                "question_ids": [
                    int(ex["question_id"]) for ex in row["examples"]
                ],
                "image_ids": [
                    int(ex["image_id"]) for ex in row["examples"]
                ],
                "reason": "empty_normalized_canonical_answer",
                "decision_id": "A004",
            })

        for row in audit["mixed_answer_type_pairs"]:
            qids = sorted(map(int, row["question_ids"]))
            exclusions.append({
                "pair_id": f"{split}:{qids[0]}:{qids[1]}",
                "source_split": split,
                "question_ids": qids,
                "image_ids": [
                    int(ex["image_id"]) for ex in row["examples"]
                ],
                "reason": "mixed_official_answer_types",
                "decision_id": "A004",
            })

    exclusions.sort(
        key=lambda r: (r["source_split"], tuple(r["question_ids"]))
    )
    excluded_ids = {r["pair_id"] for r in exclusions}
    assert len(exclusions) == len(excluded_ids) == 161

    report5["outputs"].append(save_rows(
        "additional_exclusions_A004.jsonl", exclusions
    ))
    report5["approved_exclusion_counts"] = dict(Counter(
        r["reason"] for r in exclusions
    ))

    # Load and verify the original eligibility pools.
    old_log = ROOT / "logs/block_04a_20261031_154343_2e9b562d.json"
    assert file_sha(old_log) == (
        "78d19d8f125aade489c94c1ca23fb82212d4168153cc40f09af3193be1bacd1e"
    )
    old = json.loads(old_log.read_text())
    pools = {}

    for split in ("train2014", "val2014"):
        item = next(
            r for r in old["splits"][split]["outputs"]
            if "/natural_eligible_" in r["path"]
        )
        durable = Path(item["path"])
        local = LOCAL / "eligibility" / durable.name
        copy_checked(durable, local)
        assert file_sha(local) == item["sha256"]

        rows, removed = [], 0
        with local.open() as stream:
            for line in stream:
                row = json.loads(line)
                if row["pair_id"] in excluded_ids:
                    removed += 1
                    continue

                assert row["source_split"] == split
                assert len(row["examples"]) == 2
                assert len({
                    ex["image_id"] for ex in row["examples"]
                }) == 2
                assert all(
                    ex["agreement"] >= 8 and ex["canonical"]
                    for ex in row["examples"]
                )
                assert all(
                    ex["answer_type"] == row["answer_type"]
                    for ex in row["examples"]
                )
                assert (
                    row["examples"][0]["canonical"]
                    != row["examples"][1]["canonical"]
                )
                rows.append(row)

        assert removed == (7 if split == "train2014" else 6)
        pools[split] = rows
        report5.setdefault("post_policy_pool_counts", {})[split] = dict(
            Counter(r["answer_type"] for r in rows)
        )

    quotas = {
        "train": {"yes/no": 640, "number": 256, "other": 640},
        "dev": {"yes/no": 12, "number": 8, "other": 12},
        "test": {"yes/no": 36, "number": 24, "other": 36},
    }
    selection_recipe = {
        "seed": 6304,
        "rng": "numpy.random.default_rng",
        "numpy_version": np.__version__,
        "allocation_order": [
            "natural_train", "natural_dev", "natural_test",
            "caption_train_images", "caption_train_captions",
        ],
        "pair_type_order": ["yes/no", "number", "other"],
        "pair_sort": (
            "Ascending integer question-ID tuple before seeded permutation"
        ),
        "pair_sampling": "Without replacement by pair ID",
        "evaluation_separation": (
            "Natural test excludes all natural development images"
        ),
        "within_evaluation_suite_image_reuse": (
            "Allowed; unique-image counts are recorded"
        ),
        "caption_text_deduplication": (
            "Exact text within image; retain lowest original caption ID"
        ),
        "caption_epoch_order_rng": (
            "Separate default_rng(6304), two successive permutations"
        ),
        "quotas": quotas,
        "remaining": [
            "grounding evaluation/pools", "caption development",
            "global split/label audit", "complete dataset freeze",
        ],
    }
    save_record("configs/data_selection_recipe.json", selection_recipe)

    rng = np.random.default_rng(6304)
    selected = {}
    selected["train"] = choose_pairs(
        pools["train2014"], quotas["train"], rng
    )
    selected["dev"] = choose_pairs(
        pools["val2014"], quotas["dev"], rng
    )
    selected["test"] = choose_pairs(
        pools["val2014"], quotas["test"], rng,
        blocked_images=pair_images(selected["dev"]),
    )

    dev_images = pair_images(selected["dev"])
    test_images = pair_images(selected["test"])
    natural_train_images = pair_images(selected["train"])

    assert dev_images.isdisjoint(test_images)
    assert natural_train_images.isdisjoint(dev_images | test_images)

    for split in ("train", "dev", "test"):
        rows = [
            {**r, "experiment_split": split}
            for r in selected[split]
        ]
        assert dict(Counter(
            r["answer_type"] for r in rows
        )) == quotas[split]
        assert len({r["pair_id"] for r in rows}) == len(rows)
        assert not ({r["pair_id"] for r in rows} & excluded_ids)

        report5["outputs"].append(save_rows(
            f"natural_{split}.jsonl", rows
        ))
        report5["allocation"][f"natural_{split}"] = {
            "pairs": len(rows),
            "questions": 2 * len(rows),
            "answer_type_counts": dict(Counter(
                r["answer_type"] for r in rows
            )),
            "unique_images": len(pair_images(rows)),
        }

    del pools
    gc.collect()

    # Select 6,000 training images and three distinct captions per image.
    cap = prior["caption_eligibility"]["train2014"]
    caption_source = Path(cap["index_path"])
    local_caps = LOCAL / "eligibility" / caption_source.name
    copy_checked(caption_source, local_caps)
    assert file_sha(local_caps) == cap["index_sha256"]

    by_image = defaultdict(dict)
    all_train_images = set()
    with local_caps.open() as stream:
        for line in stream:
            row = json.loads(line)
            image_id = int(row["image_id"])
            all_train_images.add(image_id)

            if row["within_content_cap"]:
                assert not row["empty_or_whitespace"]
                old_row = by_image[image_id].get(row["caption"])
                if (
                    old_row is None
                    or int(row["caption_id"]) < int(old_row["caption_id"])
                ):
                    by_image[image_id][row["caption"]] = row

    available = sorted(
        im for im, captions in by_image.items()
        if len(captions) >= 3
    )
    assert len(available) >= 6000
    assert natural_train_images <= all_train_images
    assert (dev_images | test_images).isdisjoint(all_train_images)

    caption_images = [
        available[int(i)]
        for i in rng.permutation(len(available))[:6000]
    ]
    caption_rows = []
    for image_id in sorted(caption_images):
        options = sorted(
            by_image[image_id].values(),
            key=lambda r: int(r["caption_id"]),
        )
        for index in rng.permutation(len(options))[:3]:
            caption_rows.append({
                **options[int(index)],
                "experiment_split": "train",
            })

    caption_rows.sort(
        key=lambda r: (int(r["image_id"]), int(r["caption_id"]))
    )
    assert len(caption_rows) == len({
        r["caption_id"] for r in caption_rows
    }) == 18000
    assert len({r["image_id"] for r in caption_rows}) == 6000
    assert all(
        r["within_content_cap"] and r["content_tokens"] <= 64
        for r in caption_rows
    )
    per_image = Counter(r["image_id"] for r in caption_rows)
    assert set(per_image.values()) == {3}

    report5["outputs"].append(save_rows(
        "captions_train.jsonl", caption_rows
    ))

    epoch_rng = np.random.default_rng(6304)
    epoch_orders = {
        "seed": 6304,
        "epoch_caption_ids": [
            [
                caption_rows[int(i)]["caption_id"]
                for i in epoch_rng.permutation(len(caption_rows))
            ]
            for _ in range(2)
        ],
    }
    for order in epoch_orders["epoch_caption_ids"]:
        assert len(order) == len(set(order)) == 18000

    report5["stage1_order_sha256"] = save_record(
        "manifests/selected/stage1_caption_epoch_orders.json",
        epoch_orders,
    )
    report5["allocation"]["captions_train"] = {
        "images": 6000,
        "captions": 18000,
        "passes": 2,
        "planned_presentations": 36000,
    }

    union = sorted(set(caption_images) | natural_train_images)
    union_record = {
        "source_split": "train2014",
        "image_ids": union,
        "caption_image_ids": sorted(caption_images),
        "natural_vqa_image_ids": sorted(natural_train_images),
        "purpose": (
            "Approved candidate image universe for grounding training pools"
        ),
        "grounding_pool_feasibility_verified": False,
    }
    report5["training_union_sha256"] = save_record(
        "manifests/selected/training_image_union.json",
        union_record,
    )

    reservation = {
        "natural_dev_image_ids": sorted(dev_images),
        "natural_test_image_ids": sorted(test_images),
        "exclude_from_all_grounding_evaluation_and_caption_dev":
            sorted(dev_images | test_images),
        "caption_dev_not_selected": True,
        "grounding_eval_not_selected": True,
    }
    save_record(
        "manifests/selected/evaluation_image_reservations.json",
        reservation,
    )

    report5.update({
        "status": "passed",
        "training_image_union_count": len(union),
        "estimated_training_union_feature_bytes_without_headers":
            len(union) * 196 * 768 * 2,
        "natural_dev_test_image_disjoint": True,
        "local_disk_free_GiB": round(
            shutil.disk_usage("/content").free / 2**30, 2
        ),
    })

except Exception:
    report5["status"] = "failed"
    report5["traceback"] = traceback.format_exc()
    save_record(f"logs/block_05a_{stamp}.json", report5)
    print(
        "BLOCK 5A FAILED; diagnostics and existing manifests preserved.",
        flush=True,
    )
    raise

log_path = f"logs/block_05a_{stamp}.json"
digest = save_record(log_path, report5)
print(
    "\nBLOCK 5A COMPLETE — natural subsets and training captions selected; "
    "complete dataset not frozen."
)
print(json.dumps(report5, indent=2))
print("Saved log:", ROOT / log_path)
print("SHA256:", digest)
print("Keep accelerator None. Paste the complete output.")

In [ ]:
# Block 5B: grounding pools, evaluation pair metadata, caption development.
# CPU only. No images/model weights downloaded; no questions rendered yet.
import os, json, uuid, shutil, traceback, gc
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter, defaultdict
import numpy as np

needed = (
    "ROOT", "LOCAL", "save_record", "file_sha", "copy_checked", "save_rows"
)
if any(n not in globals() for n in needed):
    raise RuntimeError("Notebook state missing. Stop; do not rerun Block 1.")

old_path = ROOT / "logs/block_05a_20261031_155959_c7d3095c.json"
assert file_sha(old_path) == (
    "a84f2436281e657665850dfad2750840b55678b271f4c1dffddc7f8e195973fc"
)
old5 = json.loads(old_path.read_text())

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    print("CPU preparation: requesting GPU release.", flush=True)
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Reconnect with accelerator None.")

stamp = (
    datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y%m%d_%H%M%S_")
    + uuid.uuid4().hex[:8]
)
report5b = {
    "attempt": stamp,
    "status": "started",
    "gpu_used": False,
    "complete_dataset_frozen": False,
    "questions_rendered": False,
    "visual_label_audit_complete": False,
    "outputs": [],
    "pool_summary": {},
}

def counting_combinations(category):
    return [
        (a, b)
        for a in range(1, 5)
        for b in range(a + 1, 5)
        if category["count_images"][str(a)]
        and category["count_images"][str(b)]
    ]

def capacity(category, kind):
    if kind == "presence":
        return len(category["present"]) * len(category["absent"])
    return sum(
        len(category["count_images"][str(a)])
        * len(category["count_images"][str(b)])
        for a, b in counting_combinations(category)
    )

def filter_catalog(catalog, banned):
    banned = set(banned)
    return [
        {
            **c,
            "present": [i for i in c["present"] if i not in banned],
            "absent": [i for i in c["absent"] if i not in banned],
            "crowd_images": [
                i for i in c["crowd_images"] if i not in banned
            ],
            "count_images": {
                k: [i for i in v if i not in banned]
                for k, v in c["count_images"].items()
            },
        }
        for c in catalog
    ]

def allocate_grounding(catalog, suite, quotas, rng):
    used, usage, rows = set(), Counter(), []

    for kind in ("presence", "counting"):
        for _ in range(quotas[kind]):
            available = [
                c for c in catalog
                if capacity(c, kind) > usage[(kind, c["category_id"])]
            ]
            if not available:
                raise RuntimeError(
                    f"Insufficient unique {kind} pairs for {suite}."
                )
            category = available[int(rng.integers(len(available)))]

            for retry in range(10000):
                if kind == "presence":
                    im1 = int(rng.choice(category["present"]))
                    im2 = int(rng.choice(category["absent"]))
                    targets, counts = ("yes", "no"), (None, None)
                else:
                    combos = counting_combinations(category)
                    a, b = combos[int(rng.integers(len(combos)))]
                    im1 = int(rng.choice(
                        category["count_images"][str(a)]
                    ))
                    im2 = int(rng.choice(
                        category["count_images"][str(b)]
                    ))
                    targets, counts = (str(a), str(b)), (a, b)

                key = (
                    kind, category["category_id"],
                    min(im1, im2), max(im1, im2),
                )
                if key not in used:
                    break
            else:
                raise RuntimeError(
                    "Unique-pair retry limit reached; "
                    "no seed/settings changed."
                )

            assert im1 != im2
            used.add(key)
            usage[(kind, category["category_id"])] += 1

            pair_id = (
                f"grounding:{suite}:{kind}:{category['category_id']}:"
                f"{min(im1, im2)}:{max(im1, im2)}"
            )
            examples = [
                {
                    "question_id": f"{pair_id}:{j}",
                    "image_id": im,
                    "target": target,
                    "canonical": target,
                    "annotated_count": count,
                }
                for j, (im, target, count) in enumerate(zip(
                    (im1, im2), targets, counts
                ))
            ]
            rows.append({
                "pair_id": pair_id,
                "source_split": "val2014",
                "experiment_split": suite,
                "question_type": kind,
                "category_id": category["category_id"],
                "category_name": category["category_name"],
                "question_template": (
                    "Does this image contain [object]?"
                    if kind == "presence"
                    else "How many [objects] can you see?"
                ),
                "question_rendered": False,
                "examples": examples,
            })

    assert dict(Counter(
        r["question_type"] for r in rows
    )) == quotas
    return rows

def image_set(rows):
    return {
        int(ex["image_id"])
        for row in rows
        for ex in row["examples"]
    }

try:
    union_path = ROOT / "manifests/selected/training_image_union.json"
    assert file_sha(union_path) == old5["training_union_sha256"]
    training_ids = set(
        json.loads(union_path.read_text())["image_ids"]
    )
    assert len(training_ids) == 8733

    natural = {}
    for suite in ("dev", "test"):
        item = next(
            r for r in old5["outputs"]
            if r["path"].endswith(f"/natural_{suite}.jsonl")
        )
        path = Path(item["path"])
        assert file_sha(path) == item["sha256"]
        with path.open() as stream:
            natural[suite] = [json.loads(line) for line in stream]

    reserved = image_set(natural["dev"]) | image_set(natural["test"])
    assert len(reserved) == 256
    assert training_ids.isdisjoint(reserved)

    coco = json.loads((
        ROOT / "manifests/annotation_downloads/coco2014.json"
    ).read_text())
    raw_hashes = {
        r["filename"]: r["sha256"] for r in coco["json_files"]
    }

    catalogs, source_ids = {}, {}
    categories_reference = None

    for split in ("train2014", "val2014"):
        print(f"Indexing {split} object annotations...", flush=True)
        path = LOCAL / "annotations" / f"instances_{split}.json"
        assert file_sha(path) == raw_hashes[path.name]
        with path.open() as stream:
            payload = json.load(stream)

        source_ids[split] = {
            int(im["id"]) for im in payload["images"]
        }
        wanted = (
            training_ids
            if split == "train2014"
            else source_ids[split] - reserved
        )
        assert wanted <= source_ids[split]

        categories = sorted(
            payload["categories"], key=lambda c: int(c["id"])
        )
        category_signature = [
            (int(c["id"]), c["name"]) for c in categories
        ]
        if categories_reference is None:
            categories_reference = category_signature
        assert category_signature == categories_reference
        assert len(categories) == 80

        present = defaultdict(set)
        crowds = defaultdict(set)
        counts = defaultdict(Counter)

        for annotation in payload["annotations"]:
            image_id = int(annotation["image_id"])
            category_id = int(annotation["category_id"])
            if image_id not in wanted:
                continue

            present[category_id].add(image_id)
            if annotation.get("iscrowd", 0):
                crowds[category_id].add(image_id)
            else:
                counts[category_id][image_id] += 1

        catalog = []
        for category in categories:
            cid = int(category["id"])
            catalog.append({
                "category_id": cid,
                "category_name": category["name"],
                "present": sorted(present[cid]),
                "absent": sorted(wanted - present[cid]),
                "crowd_images": sorted(crowds[cid]),
                "count_images": {
                    str(k): sorted(
                        im for im, n in counts[cid].items()
                        if n == k and im not in crowds[cid]
                    )
                    for k in range(1, 5)
                },
            })

        catalogs[split] = catalog
        details = [
            {
                "category_id": c["category_id"],
                "category_name": c["category_name"],
                "positive_images": len(c["present"]),
                "negative_images": len(c["absent"]),
                "crowd_image_categories_excluded_from_counting":
                    len(c["crowd_images"]),
                "count_group_sizes": {
                    k: len(v) for k, v in c["count_images"].items()
                },
                "possible_presence_pairs": capacity(c, "presence"),
                "possible_counting_pairs": capacity(c, "counting"),
            }
            for c in catalog
        ]

        detail_rel = (
            f"manifests/eligibility/grounding_pool_support_{split}.json"
        )
        detail_hash = save_record(detail_rel, details)
        pool_summary = {
            "candidate_images": len(wanted),
            "presence_eligible_categories": sum(
                capacity(c, "presence") > 0 for c in catalog
            ),
            "counting_eligible_categories": sum(
                capacity(c, "counting") > 0 for c in catalog
            ),
            "presence_unavailable_categories": [
                c["category_name"] for c in catalog
                if not capacity(c, "presence")
            ],
            "counting_unavailable_categories": [
                c["category_name"] for c in catalog
                if not capacity(c, "counting")
            ],
            "category_details_path": str(ROOT / detail_rel),
            "category_details_sha256": detail_hash,
        }
        assert pool_summary["presence_eligible_categories"]
        assert pool_summary["counting_eligible_categories"]
        report5b["pool_summary"][split] = pool_summary
        print(json.dumps({split: pool_summary}, indent=2), flush=True)

        del payload, present, crowds, counts
        gc.collect()

    assert source_ids["train2014"].isdisjoint(source_ids["val2014"])

    training_pool_record = {
        "source_split": "train2014",
        "image_ids": sorted(training_ids),
        "categories": catalogs["train2014"],
        "presence_policy": (
            "Any category annotation is positive; "
            "no category annotation is negative"
        ),
        "counting_policy": (
            "Non-crowd instance counts 1-4; exclude image-category "
            "if any crowd annotation exists"
        ),
    }
    report5b["grounding_training_pools_sha256"] = save_record(
        "manifests/selected/grounding_training_pools.json",
        training_pool_record,
    )

    recipe = {
        "seed": 6304,
        "rng": (
            "Independent numpy.random.default_rng(6304) "
            "for this allocation"
        ),
        "order": [
            "grounding_dev_presence", "grounding_dev_counting",
            "grounding_test_presence", "grounding_test_counting",
            "caption_dev_images", "caption_dev_caption",
        ],
        "category_sampling": (
            "Uniform among categories with remaining feasible pairs"
        ),
        "count_combinations": (
            "Uniform among available unordered combinations "
            "of different counts 1-4"
        ),
        "image_sampling": "Uniform within selected category/label group",
        "pair_uniqueness": (
            "No duplicate question-type/category/unordered-image "
            "pair within a suite"
        ),
        "pair_order": (
            "Presence positive then negative; "
            "counting lower then higher count"
        ),
        "image_separation": (
            "All evaluation sets mutually image-disjoint; "
            "within-set reuse allowed"
        ),
        "train_pool_image_universe_changed": False,
        "question_rendering_and_visual_label_audit_pending": True,
    }
    save_record("configs/grounding_pool_and_allocation_recipe.json", recipe)

    rng = np.random.default_rng(6304)
    ground = {}
    ground["dev"] = allocate_grounding(
        catalogs["val2014"], "dev",
        {"presence": 12, "counting": 12}, rng,
    )
    ground["test"] = allocate_grounding(
        filter_catalog(catalogs["val2014"], image_set(ground["dev"])),
        "test", {"presence": 32, "counting": 32}, rng,
    )

    for suite in ("dev", "test"):
        report5b["outputs"].append(save_rows(
            f"grounding_{suite}_pair_metadata.jsonl",
            ground[suite],
        ))

    blocked = (
        reserved | image_set(ground["dev"]) | image_set(ground["test"])
    )
    prior4_path = ROOT / "logs/block_04b_20261031_155026_82d3640b.json"
    assert file_sha(prior4_path) == (
        "35c8314906b0f5cdab93f998159f43b493fc2e111cc058cea957b2053dd23945"
    )
    cap_info = json.loads(
        prior4_path.read_text()
    )["caption_eligibility"]["val2014"]
    durable = Path(cap_info["index_path"])
    local = LOCAL / "eligibility" / durable.name
    copy_checked(durable, local)
    assert file_sha(local) == cap_info["index_sha256"]

    by_image = defaultdict(dict)
    with local.open() as stream:
        for line in stream:
            row = json.loads(line)
            image_id = int(row["image_id"])
            if image_id not in blocked and row["within_content_cap"]:
                assert not row["empty_or_whitespace"]
                old_row = by_image[image_id].get(row["caption"])
                if (
                    old_row is None
                    or int(row["caption_id"]) < int(old_row["caption_id"])
                ):
                    by_image[image_id][row["caption"]] = row

    available = sorted(by_image)
    if len(available) < 96:
        raise RuntimeError(
            "Insufficient caption-dev images after evaluation exclusions."
        )
    chosen_ids = [
        available[int(i)]
        for i in rng.permutation(len(available))[:96]
    ]
    caption_dev = []
    for image_id in sorted(chosen_ids):
        options = sorted(
            by_image[image_id].values(),
            key=lambda r: int(r["caption_id"]),
        )
        caption_dev.append({
            **options[int(rng.integers(len(options)))],
            "experiment_split": "dev",
        })

    report5b["outputs"].append(save_rows(
        "captions_dev.jsonl", caption_dev
    ))

    groups = {
        "training_union": training_ids,
        "natural_dev": image_set(natural["dev"]),
        "natural_test": image_set(natural["test"]),
        "grounding_dev": image_set(ground["dev"]),
        "grounding_test": image_set(ground["test"]),
        "captions_dev": set(chosen_ids),
    }
    names = list(groups)
    for i, left in enumerate(names):
        for right in names[i + 1:]:
            assert groups[left].isdisjoint(groups[right]), (
                f"Image overlap: {left}, {right}"
            )
    assert all(
        groups[n] <= source_ids["val2014"]
        for n in names if n != "training_union"
    )
    assert len(caption_dev) == len(set(chosen_ids)) == 96

    check_record = {
        "image_ids_by_set": {
            n: sorted(ids) for n, ids in groups.items()
        },
        "all_sets_mutually_image_disjoint": True,
        "grounding_pair_metadata_not_visually_audited": True,
        "complete_dataset_frozen": False,
    }
    report5b["global_image_allocation_sha256"] = save_record(
        "manifests/selected/global_image_allocation.json",
        check_record,
    )

    evaluation_ids = sorted(set().union(*(
        groups[n] for n in names if n != "training_union"
    )))
    report5b["candidate_cache_image_ids_sha256"] = save_record(
        "manifests/selected/candidate_cache_image_ids.json",
        {
            "train2014": sorted(training_ids),
            "val2014": evaluation_ids,
            "status": (
                "Candidate IDs only; label audit and question rendering pending"
            ),
        },
    )
    report5b.update({
        "status": "passed",
        "all_sets_mutually_image_disjoint": True,
        "allocation": {
            suite: {
                "pairs": len(rows),
                "questions": 2 * len(rows),
                "type_counts": dict(Counter(
                    r["question_type"] for r in rows
                )),
                "unique_images": len(image_set(rows)),
            }
            for suite, rows in ground.items()
        },
        "caption_dev_images": 96,
        "candidate_cache_images": {
            "train2014": len(training_ids),
            "val2014": len(evaluation_ids),
        },
        "local_disk_free_GiB": round(
            shutil.disk_usage("/content").free / 2**30, 2
        ),
    })

except Exception:
    report5b["status"] = "failed"
    report5b["traceback"] = traceback.format_exc()
    save_record(f"logs/block_05b_{stamp}.json", report5b)
    print(
        "BLOCK 5B FAILED; diagnostics and earlier allocations retained.",
        flush=True,
    )
    raise

log_path = f"logs/block_05b_{stamp}.json"
digest = save_record(log_path, report5b)
print(
    "\nBLOCK 5B COMPLETE — grounding pools and evaluation allocation; "
    "label audit still pending."
)
print(json.dumps(report5b, indent=2))
print("Saved log:", ROOT / log_path)
print("SHA256:", digest)
print("Keep accelerator None. Paste the complete output.")

In [ ]:
# Block 6A: approved protocol, rendered wording, and audit preparation.
# CPU only. No images, model weights, calibration computation, or training.
import os, json, uuid, shutil, traceback, math, csv, io
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter
import numpy as np

needed = (
    "ROOT", "LOCAL", "save_record", "file_sha", "copy_checked", "save_rows",
    "tokenizer", "capacity", "counting_combinations"
)
if any(n not in globals() for n in needed):
    raise RuntimeError("Notebook state missing. Stop; do not rerun Block 1.")

previous = ROOT / "logs/block_05b_20261031_161050_c9dc05e9.json"
assert file_sha(previous) == (
    "21ccdac5191bebd4caeba690adf481ecb47ef5ae476a7e03127731bb9ca5499a"
)
old5b = json.loads(previous.read_text())

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    print("CPU preparation: requesting GPU release.", flush=True)
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Reconnect with accelerator None.")

stamp = (
    datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y%m%d_%H%M%S_")
    + uuid.uuid4().hex[:8]
)
report6 = {
    "attempt": stamp,
    "status": "started",
    "gpu_used": False,
    "complete_dataset_frozen": False,
    "visual_labels_reviewed": False,
    "calibration_computed": False,
    "outputs": [],
    "token_maxima": {},
}

SPEC = json.loads(r'''{"status":"approved","date":"2026-10-31","scope":"Exact values previously left unspecified; extends approved directions without replacing earlier approvals","wording":{"caption_user_text":"Describe this image.","vqa_user_text":"Answer with a short phrase.\n{original_question}","system_message":null,"thinking_enabled":false,"presence_question":"Does this image contain {presence_phrase}?","counting_question":"How many {counting_phrase} can you see?","same_vqa_text_for_all_image_conditions":true,"preserve_natural_questions_and_original_targets":true,"cap_check":"Check original question and complete rendered VQA user text each <=80 tokens. Stop without truncation or reselection if any chosen example fails."},"audit":{"seed":6304,"evaluation_grounding_pairs":88,"training_presence_pairs":80,"training_counting_pairs":78,"total_pairs":246,"training_sample_rule":"One complete pair for every eligible category/task, using an independent default_rng(6304); uniform eligible count combination and images","review":"Full images plus category boxes; review annotation absence, omitted objects, count/group ambiguity, occlusion and crowd handling independently of model predictions","dispositions":["accept","ambiguous","annotation_error"],"automatic_replacement_or_relabeling":false,"training_budget_affected":false},"calibration":{"seed":6304,"training_images":256,"training_caption_examples":256,"selection":"Sorted IDs, independent default_rng(6304), without replacement; image sample from 8733-image training union; captions from selected 18000","numerator":"Global RMS of frozen Qwen embeddings of all content tokens in sampled original captions; exclude chat, termination and padding tokens","denominator":"Global RMS of initial seeded connector outputs for all 196 patches of sampled cached training images","arithmetic":"FP32 streaming sum of squares and element counts","scale":"numerator / denominator; error if either RMS is zero or nonfinite","placement":"One fixed non-trainable scalar after connector MLP, before conversion to Qwen FP16 input embeddings; retain throughout all stages","saved_evidence":["sample IDs","both RMS values","scale","initial connector state and hash"],"evaluation_inputs_used":false,"additional_normalization_or_clamping":false},"learning_rate_endpoints":{"warmup_updates":[113,75,38],"update_index":"One-based successful optimizer update u=1..U within stage","warmup":"LR(u)=peak*u/w for 1<=u<=w","decay":"LR(u)=peak*(0.1+0.9*(1+cos(pi*(u-w)/(U-w)))/2) for w<u<=U","first":"peak/w","warmup_end":"peak","stage_end":"0.1*peak","applied":"Set the LR for update u before its optimizer step; only completed updates advance state"},"decision_id":"A005","approval_date":"2026-10-31","user_approval":"approved and document and let's continue"}''')

# Literal approved noun forms: category | presence phrase | counting phrase.
NOUN_FORMS = """person|a person|people
bicycle|a bicycle|bicycles
car|a car|cars
motorcycle|a motorcycle|motorcycles
airplane|an airplane|airplanes
bus|a bus|buses
train|a train|trains
truck|a truck|trucks
boat|a boat|boats
traffic light|a traffic light|traffic lights
fire hydrant|a fire hydrant|fire hydrants
stop sign|a stop sign|stop signs
parking meter|a parking meter|parking meters
bench|a bench|benches
bird|a bird|birds
cat|a cat|cats
dog|a dog|dogs
horse|a horse|horses
sheep|a sheep|sheep
cow|a cow|cows
elephant|an elephant|elephants
bear|a bear|bears
zebra|a zebra|zebras
giraffe|a giraffe|giraffes
backpack|a backpack|backpacks
umbrella|an umbrella|umbrellas
handbag|a handbag|handbags
tie|a tie|ties
suitcase|a suitcase|suitcases
frisbee|a frisbee|frisbees
skis|skis|skis
snowboard|a snowboard|snowboards
sports ball|a sports ball|sports balls
kite|a kite|kites
baseball bat|a baseball bat|baseball bats
baseball glove|a baseball glove|baseball gloves
skateboard|a skateboard|skateboards
surfboard|a surfboard|surfboards
tennis racket|a tennis racket|tennis rackets
bottle|a bottle|bottles
wine glass|a wine glass|wine glasses
cup|a cup|cups
fork|a fork|forks
knife|a knife|knives
spoon|a spoon|spoons
bowl|a bowl|bowls
banana|a banana|bananas
apple|an apple|apples
sandwich|a sandwich|sandwiches
orange|an orange|oranges
broccoli|broccoli|broccoli
carrot|a carrot|carrots
hot dog|a hot dog|hot dogs
pizza|pizza|pizzas
donut|a donut|donuts
cake|cake|cakes
chair|a chair|chairs
couch|a couch|couches
potted plant|a potted plant|potted plants
bed|a bed|beds
dining table|a dining table|dining tables
toilet|a toilet|toilets
tv|a TV|TVs
laptop|a laptop|laptops
mouse|a mouse|mice
remote|a remote|remotes
keyboard|a keyboard|keyboards
cell phone|a cell phone|cell phones
microwave|a microwave|microwaves
oven|an oven|ovens
toaster|a toaster|toasters
sink|a sink|sinks
refrigerator|a refrigerator|refrigerators
book|a book|books
clock|a clock|clocks
vase|a vase|vases
scissors|scissors|scissors
teddy bear|a teddy bear|teddy bears
hair drier|a hair drier|hair driers
toothbrush|a toothbrush|toothbrushes"""

SPEC["noun_forms"] = {}
for line in NOUN_FORMS.splitlines():
    name, presence, counting = line.split("|")
    assert name not in SPEC["noun_forms"]
    SPEC["noun_forms"][name] = {
        "presence_phrase": presence,
        "counting_phrase": counting,
    }
assert len(SPEC["noun_forms"]) == 80

def load_rows(item):
    durable = Path(item["path"])
    local = LOCAL / "protocol_inputs" / durable.name
    copy_checked(durable, local)
    assert file_sha(local) == item["sha256"]
    with local.open() as stream:
        return [json.loads(line) for line in stream]

def user_text(question):
    return SPEC["wording"]["vqa_user_text"].format(
        original_question=question
    )

violations = []

def length(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def checked_question(question, identity):
    text = user_text(question)
    qn, un = length(question), length(text)
    maxima = report6["token_maxima"]
    maxima["original_question"] = max(
        maxima.get("original_question", 0), qn
    )
    maxima["complete_vqa_user_text"] = max(
        maxima.get("complete_vqa_user_text", 0), un
    )
    if qn > 80 or un > 80:
        violations.append({
            "id": identity,
            "question_tokens": qn,
            "user_text_tokens": un,
        })
    return {
        "question": question,
        "user_text": text,
        "question_content_tokens": qn,
        "user_content_tokens": un,
    }

def format_pair(row, question):
    formatted = {
        **row,
        **checked_question(question, row["pair_id"]),
    }
    for ex in row["examples"]:
        n = length(ex["target"])
        report6["token_maxima"]["short_answer_target"] = max(
            report6["token_maxima"].get("short_answer_target", 0), n
        )
        if n > 16:
            violations.append({
                "id": ex["question_id"],
                "answer_tokens": n,
            })
    return formatted

def ground_question(category_name, kind):
    forms = SPEC["noun_forms"][category_name]
    if kind == "presence":
        return SPEC["wording"]["presence_question"].format(
            presence_phrase=forms["presence_phrase"]
        )
    if kind == "counting":
        return SPEC["wording"]["counting_question"].format(
            counting_phrase=forms["counting_phrase"]
        )
    raise ValueError(kind)

def draw_training_audit_pair(category, kind, rng):
    if kind == "presence":
        images = (
            int(rng.choice(category["present"])),
            int(rng.choice(category["absent"])),
        )
        targets, counts = ("yes", "no"), (None, None)
    else:
        combos = counting_combinations(category)
        a, b = combos[int(rng.integers(len(combos)))]
        images = (
            int(rng.choice(category["count_images"][str(a)])),
            int(rng.choice(category["count_images"][str(b)])),
        )
        targets, counts = (str(a), str(b)), (a, b)

    pair_id = (
        f"grounding:train_audit:{kind}:{category['category_id']}:"
        f"{min(images)}:{max(images)}"
    )
    return {
        "pair_id": pair_id,
        "source_split": "train2014",
        "experiment_split": "train_audit",
        "audit_only": True,
        "question_type": kind,
        "category_id": category["category_id"],
        "category_name": category["category_name"],
        "question_rendered": True,
        "examples": [
            {
                "question_id": f"{pair_id}:{j}",
                "image_id": im,
                "target": target,
                "canonical": target,
                "annotated_count": count,
            }
            for j, (im, target, count) in enumerate(zip(
                images, targets, counts
            ))
        ],
    }

def learning_rate_for_update(peak, updates, warmup, update):
    if not (
        peak > 0 and 0 < warmup < updates
        and 1 <= update <= updates
    ):
        raise ValueError("Invalid schedule input.")
    if update <= warmup:
        return peak * update / warmup
    return peak * (
        0.1 + 0.9 * (
            1 + math.cos(
                math.pi * (update - warmup) / (updates - warmup)
            )
        ) / 2
    )

try:
    report6["approved_protocol_sha256"] = save_record(
        "configs/exact_protocol_A005.json", SPEC
    )
    pool_path = ROOT / "manifests/selected/grounding_training_pools.json"
    assert file_sha(pool_path) == old5b["grounding_training_pools_sha256"]
    pool = json.loads(pool_path.read_text())
    categories = sorted(
        pool["categories"], key=lambda c: c["category_id"]
    )
    assert {
        c["category_name"] for c in categories
    } == set(SPEC["noun_forms"])
    assert sum(capacity(c, "presence") > 0 for c in categories) == 80
    assert sum(capacity(c, "counting") > 0 for c in categories) == 78

    bank = []
    for category in categories:
        for kind in ("presence", "counting"):
            bank.append({
                "category_id": category["category_id"],
                "category_name": category["category_name"],
                "question_type": kind,
                "available_in_training_pool":
                    capacity(category, kind) > 0,
                **checked_question(
                    ground_question(category["category_name"], kind),
                    f"bank:{category['category_id']}:{kind}",
                ),
            })

    output_sets = {}
    for suite in ("dev", "test"):
        item = next(
            r for r in old5b["outputs"]
            if r["path"].endswith(
                f"/grounding_{suite}_pair_metadata.jsonl"
            )
        )
        metadata_rows = load_rows(item)
        output_sets[f"grounding_{suite}.jsonl"] = [
            {
                **format_pair(
                    row,
                    ground_question(
                        row["category_name"], row["question_type"]
                    ),
                ),
                "question_rendered": True,
            }
            for row in metadata_rows
        ]

    old5a_path = ROOT / "logs/block_05a_20261031_155959_c7d3095c.json"
    assert file_sha(old5a_path) == (
        "a84f2436281e657665850dfad2750840b55678b271f4c1dffddc7f8e195973fc"
    )
    old5a = json.loads(old5a_path.read_text())

    for suite in ("train", "dev", "test"):
        item = next(
            r for r in old5a["outputs"]
            if r["path"].endswith(f"/natural_{suite}.jsonl")
        )
        rows = load_rows(item)
        output_sets[f"natural_{suite}_formatted.jsonl"] = [
            format_pair(row, row["question"]) for row in rows
        ]

    # Dedicated audit RNG: never advances curriculum sampling.
    audit_rng = np.random.default_rng(6304)
    train_audit = []
    for kind in ("presence", "counting"):
        for category in categories:
            if capacity(category, kind):
                row = draw_training_audit_pair(
                    category, kind, audit_rng
                )
                train_audit.append(format_pair(
                    row,
                    ground_question(category["category_name"], kind),
                ))

    assert Counter(
        r["question_type"] for r in train_audit
    ) == {"presence": 80, "counting": 78}

    output_sets["grounding_training_audit.jsonl"] = train_audit
    audit_pairs = (
        train_audit
        + output_sets["grounding_dev.jsonl"]
        + output_sets["grounding_test.jsonl"]
    )
    assert len(audit_pairs) == len({
        r["pair_id"] for r in audit_pairs
    }) == 246

    if violations:
        report6["cap_violations"] = violations
        raise RuntimeError(
            f"{len(violations)} cap violations. "
            "No truncation or reselection performed."
        )

    report6["grounding_question_bank_sha256"] = save_record(
        "configs/grounding_question_bank.json", bank
    )
    for filename, rows in output_sets.items():
        report6["outputs"].append(save_rows(filename, rows))

    # Review fields remain blank until images are actually inspected.
    review_local = (
        LOCAL / "selection" / "annotation_audit_review_template.csv"
    )
    columns = [
        "pair_id", "source_split", "experiment_split", "question_type",
        "category_id", "category_name", "question",
        "image_0", "target_0", "image_1", "target_1",
        "label_status", "notes", "reviewer", "reviewed_at",
    ]
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(
        buffer, fieldnames=columns, lineterminator="\n"
    )
    writer.writeheader()

    for row in audit_pairs:
        a, b = row["examples"]
        writer.writerow({
            "pair_id": row["pair_id"],
            "source_split": row["source_split"],
            "experiment_split": row["experiment_split"],
            "question_type": row["question_type"],
            "category_id": row["category_id"],
            "category_name": row["category_name"],
            "question": row["question"],
            "image_0": a["image_id"],
            "target_0": a["target"],
            "image_1": b["image_id"],
            "target_1": b["target"],
            "label_status": "",
            "notes": "",
            "reviewer": "",
            "reviewed_at": "",
        })

    if review_local.exists():
        assert review_local.read_text() == buffer.getvalue(), (
            "Existing review template differs; preserved."
        )
    else:
        review_local.write_text(buffer.getvalue())

    review_target = ROOT / "audits/annotation_audit_review_template.csv"
    report6["audit_review_template_sha256"] = copy_checked(
        review_local, review_target
    )

    cache_path = ROOT / "manifests/selected/candidate_cache_image_ids.json"
    assert file_sha(cache_path) == old5b["candidate_cache_image_ids_sha256"]
    cache_ids = json.loads(cache_path.read_text())
    audit_ids = {
        split: sorted({
            ex["image_id"]
            for row in audit_pairs
            if row["source_split"] == split
            for ex in row["examples"]
        })
        for split in ("train2014", "val2014")
    }
    assert all(
        set(audit_ids[s]) <= set(cache_ids[s]) for s in audit_ids
    )

    report6["audit_image_ids_sha256"] = save_record(
        "audits/annotation_audit_image_ids.json", audit_ids
    )
    report6["audit_plan_sha256"] = save_record(
        "audits/annotation_audit_plan.json",
        {
            "decision_id": "A005",
            "seed": 6304,
            "training_pairs": 158,
            "evaluation_pairs": 88,
            "total_pairs": 246,
            "image_presentations": 492,
            "image_ids": audit_ids,
            "review_template": str(review_target),
            "completed_reviews": 0,
            "allowed_statuses": [
                "accept", "ambiguous", "annotation_error"
            ],
            "automatic_changes_allowed": False,
            "model_predictions_used": False,
            "audit_views_do_not_count_as_training_presentations": True,
        },
    )

    # Select calibration IDs only; no weights loaded or scale computed.
    union_path = ROOT / "manifests/selected/training_image_union.json"
    assert file_sha(union_path) == old5a["training_union_sha256"]
    training_images = sorted(
        json.loads(union_path.read_text())["image_ids"]
    )
    item = next(
        r for r in old5a["outputs"]
        if r["path"].endswith("/captions_train.jsonl")
    )
    caption_rows = load_rows(item)
    caption_ids = sorted(r["caption_id"] for r in caption_rows)
    assert len(training_images) == 8733
    assert len(caption_ids) == len(set(caption_ids)) == 18000

    calibration_rng = np.random.default_rng(6304)
    calibration_ids = {
        "decision_id": "A005",
        "seed": 6304,
        "sampling_order": [
            "training_images", "training_caption_examples"
        ],
        "image_ids": [
            training_images[int(i)]
            for i in calibration_rng.permutation(
                len(training_images)
            )[:256]
        ],
        "caption_ids": [
            caption_ids[int(i)]
            for i in calibration_rng.permutation(
                len(caption_ids)
            )[:256]
        ],
        "scale_computed": False,
    }
    assert (
        len(set(calibration_ids["image_ids"]))
        == len(set(calibration_ids["caption_ids"]))
        == 256
    )
    report6["calibration_sample_sha256"] = save_record(
        "configs/calibration_sample_ids.json", calibration_ids
    )

    caption_prompt_tokens = length(
        SPEC["wording"]["caption_user_text"]
    )
    assert caption_prompt_tokens <= 80
    report6["caption_user_text_tokens"] = caption_prompt_tokens

    # Check approved LR definitions; perform no optimizer updates.
    schedule_cases = [
        (1, "connector", 2250, 113, 1e-3),
        (2, "connector", 1500, 75, 1e-4),
        (2, "lora", 1500, 75, 2e-5),
        (3, "connector", 750, 38, 5e-5),
        (3, "lora", 750, 38, 1e-5),
    ]
    endpoints = []
    for stage, component, updates, warmup, peak in schedule_cases:
        values = [
            learning_rate_for_update(peak, updates, warmup, u)
            for u in range(1, updates + 1)
        ]
        assert math.isclose(values[0], peak / warmup, rel_tol=1e-12)
        assert math.isclose(values[warmup - 1], peak, rel_tol=1e-12)
        assert math.isclose(values[-1], 0.1 * peak, rel_tol=1e-12)
        assert all(
            a <= b
            for a, b in zip(values[:warmup - 1], values[1:warmup])
        )
        assert all(
            a >= b
            for a, b in zip(values[warmup - 1:-1], values[warmup:])
        )
        endpoints.append({
            "stage": stage,
            "component": component,
            "updates": updates,
            "warmup": warmup,
            "peak": peak,
            "first": values[0],
            "warmup_end": values[warmup - 1],
            "last": values[-1],
        })

    report6["lr_endpoints_sha256"] = save_record(
        "configs/training_lr_endpoints.json", endpoints
    )
    report6.update({
        "status": "passed",
        "audit_pairs": {
            "training": 158, "evaluation": 88, "total": 246
        },
        "audit_unique_images": {
            s: len(ids) for s, ids in audit_ids.items()
        },
        "audit_reviews_completed": 0,
        "calibration_images_selected": 256,
        "calibration_captions_selected": 256,
        "lr_endpoint_checks_passed": True,
        "chat_serialization_example": tokenizer.apply_chat_template(
            [{
                "role": "user",
                "content": output_sets["grounding_dev.jsonl"][0]["user_text"],
            }],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        ),
        "local_disk_free_GiB": round(
            shutil.disk_usage("/content").free / 2**30, 2
        ),
    })

except Exception:
    report6["status"] = "failed"
    report6["traceback"] = traceback.format_exc()
    save_record(f"logs/block_06a_{stamp}.json", report6)
    print(
        "BLOCK 6A FAILED; diagnostics saved; "
        "no examples truncated or reselected.",
        flush=True,
    )
    raise

log_path = f"logs/block_06a_{stamp}.json"
digest = save_record(log_path, report6)
print(
    "\nBLOCK 6A COMPLETE — approved protocol saved; wording checked; "
    "audits prepared, not completed."
)
print(json.dumps(report6, indent=2))
print("Saved log:", ROOT / log_path)
print("SHA256:", digest)
print("Keep accelerator None. Paste the complete output.")

In [ ]:
# Block 6B: original audit images and annotation-box index. CPU only.
# No label judgments, image resizing, model weights, or GPU work.
import os, json, uuid, shutil, traceback, gc, time, urllib.request
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image

needed = ("ROOT", "LOCAL", "save_record", "file_sha", "copy_checked")
if any(n not in globals() for n in needed):
    raise RuntimeError("Notebook state missing. Stop; do not rerun Block 1.")

old_path = ROOT / "logs/block_06a_20261031_163930_3cb996af.json"
assert file_sha(old_path) == (
    "44a121f200cfd59ec38771010438328e8aaa35482989c5843ddd54c42ddbf13a"
)
old6 = json.loads(old_path.read_text())

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    print("CPU image preparation: requesting GPU release.", flush=True)
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Reconnect with accelerator None.")

stamp = (
    datetime.now(ZoneInfo("Asia/Karachi")).strftime("%Y%m%d_%H%M%S_")
    + uuid.uuid4().hex[:8]
)
started = time.monotonic()
report6b = {
    "attempt": stamp,
    "status": "started",
    "gpu_used": False,
    "visual_reviews_imported": 0,
    "complete_dataset_frozen": False,
    "model_weights_loaded": False,
    "images_verified": 0,
}

# Decode for verification; never rewrite the original pixels.
def inspect_original(path, expected):
    with Image.open(path) as image:
        image.load()
        if (
            image.format != "JPEG"
            or image.size != (expected["width"], expected["height"])
        ):
            raise IOError(f"Image format/dimensions mismatch: {path}")
        return {
            "width": image.width,
            "height": image.height,
            "mode": image.mode,
            "exif_orientation": int(image.getexif().get(274, 1)),
        }

def ensure_audit_image(job):
    local = LOCAL / "images" / job["source_split"] / job["file_name"]
    durable = ROOT / "raw_images" / job["source_split"] / job["file_name"]
    receipt_rel = (
        f"manifests/images/{job['source_split']}/"
        f"{job['image_id']:012d}.json"
    )
    receipt_path = ROOT / receipt_rel
    local.parent.mkdir(parents=True, exist_ok=True)

    if receipt_path.exists():
        receipt = json.loads(receipt_path.read_text())
        if (
            receipt["url"] != job["url"]
            or receipt["image_id"] != job["image_id"]
        ):
            raise IOError("Conflicting saved image identity.")

        if durable.exists():
            if file_sha(durable) != receipt["sha256"]:
                raise IOError(
                    f"Drive image differs from its receipt: {durable}"
                )
            copy_checked(durable, local)

        if (
            not local.exists()
            or file_sha(local) != receipt["sha256"]
        ):
            raise IOError(
                "Recorded image is missing/changed; "
                "preserve it and report the error."
            )
        assert inspect_original(local, job) == receipt["decoded_metadata"]
        copy_checked(local, durable)
        return receipt, "reused"

    temporary = local.with_name(local.name + ".download.part")
    last_error = None

    for retry in range(3):
        try:
            request = urllib.request.Request(
                job["url"],
                headers={"User-Agent": "ATML-PA3-CPU-data-preparation"},
            )
            with urllib.request.urlopen(request, timeout=30) as response:
                if not response.geturl().startswith("https://"):
                    raise IOError("Unexpected non-HTTPS redirect.")

                expected_bytes = response.headers.get("Content-Length")
                with temporary.open("wb") as out:
                    shutil.copyfileobj(response, out)

                if (
                    expected_bytes is not None
                    and temporary.stat().st_size != int(expected_bytes)
                ):
                    raise IOError("Incomplete image response.")

            decoded = inspect_original(temporary, job)
            last_error = None
            break

        except Exception as exc:
            last_error = exc
            if retry < 2:
                time.sleep(retry + 1)

    if last_error is not None:
        raise last_error

    digest = file_sha(temporary)
    if local.exists():
        if file_sha(local) != digest:
            raise IOError(
                f"Existing unrecorded image differs; preserved: {local}"
            )
        temporary.unlink()
    else:
        temporary.rename(local)

    copy_checked(local, durable)
    receipt = {
        "source_split": job["source_split"],
        "image_id": job["image_id"],
        "file_name": job["file_name"],
        "url": job["url"],
        "original_coco_url": job.get("original_coco_url"),
        "sha256": digest,
        "bytes": local.stat().st_size,
        "decoded_metadata": decoded,
        "hash_provenance": (
            "Observed original-image SHA256; HTTPS COCO object download"
        ),
    }
    save_record(receipt_rel, receipt)
    return receipt, "downloaded"

try:
    ids_path = ROOT / "audits/annotation_audit_image_ids.json"
    assert file_sha(ids_path) == old6["audit_image_ids_sha256"]
    ids = json.loads(ids_path.read_text())
    assert len(ids["train2014"]) == 314
    assert len(ids["val2014"]) == 174

    pairs = []
    for name in (
        "grounding_training_audit.jsonl",
        "grounding_dev.jsonl",
        "grounding_test.jsonl",
    ):
        item = next(
            r for r in old6["outputs"]
            if r["path"].endswith("/" + name)
        )
        path = Path(item["path"])
        assert file_sha(path) == item["sha256"]
        with path.open() as stream:
            pairs.extend(json.loads(line) for line in stream)
    assert len(pairs) == 246

    requested = {
        (
            r["source_split"],
            int(ex["image_id"]),
            int(r["category_id"]),
        )
        for r in pairs
        for ex in r["examples"]
    }
    coco = json.loads((
        ROOT / "manifests/annotation_downloads/coco2014.json"
    ).read_text())
    hashes = {
        r["filename"]: r["sha256"] for r in coco["json_files"]
    }
    jobs, box_records, image_metadata, licenses = [], {}, {}, {}

    for split in ("train2014", "val2014"):
        print("Preparing audit image metadata:", split, flush=True)
        path = LOCAL / "annotations" / f"instances_{split}.json"
        assert file_sha(path) == hashes[path.name]

        with path.open() as stream:
            payload = json.load(stream)

        wanted = set(ids[split])
        chosen = {
            int(im["id"]): im
            for im in payload["images"]
            if int(im["id"]) in wanted
        }
        assert set(chosen) == wanted
        licenses[split] = payload.get("licenses", [])
        boxes = defaultdict(list)

        for annotation in payload["annotations"]:
            key = (
                split,
                int(annotation["image_id"]),
                int(annotation["category_id"]),
            )
            if key in requested:
                boxes[key].append({
                    "annotation_id": int(annotation["id"]),
                    "bbox": annotation["bbox"],
                    "iscrowd": bool(annotation.get("iscrowd", 0)),
                })

        for key in sorted(k for k in requested if k[0] == split):
            box_records["|".join(map(str, key))] = sorted(
                boxes[key], key=lambda a: a["annotation_id"]
            )

        for image_id, im in sorted(chosen.items()):
            filename = im["file_name"]
            assert Path(filename).name == filename
            assert filename.endswith(f"{image_id:012d}.jpg")

            job = {
                "source_split": split,
                "image_id": image_id,
                "file_name": filename,
                "width": int(im["width"]),
                "height": int(im["height"]),
                "original_coco_url": im.get("coco_url"),
                "url": (
                    "https://s3.amazonaws.com/images.cocodataset.org/"
                    f"{split}/{filename}"
                ),
            }
            jobs.append(job)
            image_metadata[f"{split}|{image_id}"] = {
                **job,
                "flickr_url": im.get("flickr_url"),
                "license_id": im.get("license"),
            }

        del payload, chosen, boxes
        gc.collect()

    # Annotation consistency is separate from actual visual correctness.
    for row in pairs:
        for ex in row["examples"]:
            key = (
                f"{row['source_split']}|{ex['image_id']}|"
                f"{row['category_id']}"
            )
            source_boxes = box_records[key]
            if row["question_type"] == "presence":
                assert bool(source_boxes) == (ex["target"] == "yes")
            else:
                assert not any(a["iscrowd"] for a in source_boxes)
                assert len(source_boxes) == int(ex["target"])

    report6b["annotation_consistency_check_passed"] = True
    report6b["audit_box_index_sha256"] = save_record(
        "audits/annotation_box_index.json",
        {
            "boxes_by_split_image_category": box_records,
            "images": image_metadata,
            "licenses": licenses,
            "coordinate_system": (
                "Original encoded JPEG dimensions; "
                "boxes are COCO x,y,width,height"
            ),
            "visual_audit_complete": False,
        },
    )

    results, activities = [], Counter()

    # Check one real image from each split before the bulk requests.
    first = [
        next(j for j in jobs if j["source_split"] == split)
        for split in ("train2014", "val2014")
    ]
    first_keys = {
        (j["source_split"], j["image_id"]) for j in first
    }
    for job in first:
        print("Checking image source:", job["url"], flush=True)
        receipt, activity = ensure_audit_image(job)
        results.append(receipt)
        activities[activity] += 1
        report6b["images_verified"] = len(results)

    remaining = [
        j for j in jobs
        if (j["source_split"], j["image_id"]) not in first_keys
    ]

    # Small batches limit outstanding requests; completed files persist.
    for start in range(0, len(remaining), 32):
        with ThreadPoolExecutor(max_workers=4) as executor:
            future_map = {
                executor.submit(ensure_audit_image, j): j
                for j in remaining[start:start + 32]
            }
            for future in as_completed(future_map):
                receipt, activity = future.result()
                results.append(receipt)
                activities[activity] += 1

        report6b["images_verified"] = len(results)
        save_record(
            f"logs/image_download_progress/{stamp}_{len(results):04d}.json",
            {
                "verified_images": len(results),
                "activities": dict(activities),
                "elapsed_seconds": round(
                    time.monotonic() - started, 2
                ),
            },
        )
        print(
            f"Verified and saved {len(results)}/488 original audit images",
            flush=True,
        )

    results.sort(
        key=lambda r: (r["source_split"], r["image_id"])
    )
    assert len(results) == len({
        (r["source_split"], r["image_id"]) for r in results
    }) == 488

    report6b["image_manifest_sha256"] = save_record(
        "audits/annotation_audit_image_manifest.json", results
    )
    report6b.update({
        "status": "passed",
        "images_verified": len(results),
        "image_counts": dict(Counter(
            r["source_split"] for r in results
        )),
        "total_original_image_bytes": sum(
            r["bytes"] for r in results
        ),
        "activities_this_attempt": dict(activities),
        "nonstandard_exif_orientation_images": [
            f"{r['source_split']}|{r['image_id']}"
            for r in results
            if r["decoded_metadata"]["exif_orientation"] != 1
        ],
        "image_mode_counts": dict(Counter(
            r["decoded_metadata"]["mode"] for r in results
        )),
        "elapsed_seconds": round(
            time.monotonic() - started, 2
        ),
        "local_disk_free_GiB": round(
            shutil.disk_usage("/content").free / 2**30, 2
        ),
    })

except Exception:
    report6b["status"] = "failed"
    report6b["traceback"] = traceback.format_exc()
    save_record(f"logs/block_06b_{stamp}.json", report6b)
    print(
        "BLOCK 6B FAILED. Completed image files/receipts retained; "
        "no image IDs replaced.",
        flush=True,
    )
    raise

log_path = f"logs/block_06b_{stamp}.json"
digest = save_record(log_path, report6b)
print(
    "\nBLOCK 6B COMPLETE — audit images and box metadata verified; "
    "visual review not performed."
)
print(json.dumps(report6b, indent=2))
print("Saved log:", ROOT / log_path)
print("SHA256:", digest)
print("Keep accelerator None. Paste the complete output.")

In [ ]:
# Block 6C launcher: verify the reviewed script before executing it.
from google.colab import files
import hashlib

expected_name = "Task_2_Colab_Block_06C.py"
expected_sha256 = "750ea605fa6bee15dd7d1419a8c64ddefe1be939f745bb5c824550e58cb5dbab"

print(f"Select only {expected_name}")
uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Select exactly one file: Task_2_Colab_Block_06C.py")

name, source = next(iter(uploaded.items()))
observed_sha256 = hashlib.sha256(source).hexdigest()

if observed_sha256 != expected_sha256:
    raise RuntimeError(
        "Script checksum mismatch. Nothing executed. "
        "Download the linked Block 6C file again."
    )

print("Script checksum verified. Preparing the CPU-only audit bundle...")
exec(compile(source.decode("utf-8"), expected_name, "exec"), globals())

In [ ]:
# Block 6D — launch your interactive manual review.
from google.colab import files
import hashlib

print("Select Task_2_Colab_Block_06D_Manual_Review.py")
manual_review_upload = files.upload()

if len(manual_review_upload) != 1:
    raise RuntimeError("Select exactly the manual-review Python file.")

manual_review_name, manual_review_cell_bytes = next(
    iter(manual_review_upload.items())
)

expected_sha256 = (
    "2831955b033cbf5b6a25cdc59e594cb0db06566dea03ecbe902b850155b24a5d"
)
if hashlib.sha256(manual_review_cell_bytes).hexdigest() != expected_sha256:
    raise RuntimeError("Checksum mismatch. Nothing executed; download the script again.")

exec(
    compile(
        manual_review_cell_bytes.decode("utf-8"),
        "Task_2_Colab_Block_06D_Manual_Review.py",
        "exec",
    ),
    globals(),
)

In [ ]:
# Task 2 — seven-case manual reconciliation. Accelerator: None.
import os
import hashlib
from google.colab import files, runtime

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    try:
        runtime.unassign()
    finally:
        raise RuntimeError(
            "Reconnect with accelerator None, then run this cell again."
        )

EXPECTED_SHA256 = (
    "85569c10c48d3782cec64bf0302874219829a6872bf24c44ba05d8785515538e"
)

print("Upload Task_2_Colab_Block_06E_Reconciliation.py")
uploaded = files.upload()

matching_files = [
    content
    for content in uploaded.values()
    if hashlib.sha256(content).hexdigest() == EXPECTED_SHA256
]

if len(matching_files) != 1:
    raise ValueError(
        "The uploaded file does not match the prepared review code. "
        "Download the linked file and upload it unchanged."
    )

reconciliation_cell_bytes = matching_files[0]
print("Review-code checksum verified.")

exec(
    compile(
        reconciliation_cell_bytes,
        "Task_2_Colab_Block_06E_Reconciliation.py",
        "exec",
    ),
    globals(),
)

In [ ]:
# Task 2 — read-only metadata export. Accelerator: None.
import os
import hashlib
from google.colab import files, runtime

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Reconnect with accelerator None.")

EXPECTED_SHA256 = (
    "9c704c3bda75e60c6c77cc8136718273823df2ee30da254786b8d5f846fa007c"
)

print("Upload Task_2_Colab_Block_06F_Metadata_Export.py")
uploaded = files.upload()
matches = [
    content for content in uploaded.values()
    if hashlib.sha256(content).hexdigest() == EXPECTED_SHA256
]

if len(matches) != 1:
    raise ValueError("Upload the linked Block 6F file unchanged.")

metadata_export_cell_bytes = matches[0]
print("Block 6F checksum verified.")

exec(
    compile(
        metadata_export_cell_bytes,
        "Task_2_Colab_Block_06F_Metadata_Export.py",
        "exec",
    ),
    globals(),
)

In [ ]:
# Task 2 — prepare images and review A007 replacement candidates.
import os
import io
import hashlib
import zipfile
from google.colab import files, runtime

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    try:
        runtime.unassign()
    finally:
        raise RuntimeError("Reconnect with accelerator None.")

EXPECTED_SHA256 = (
    "8aac54f305bfd982116d083c1c4c8596a415439b0242898211d5a926f1457a0e"
)

print("Upload Task_2_Block_06G_Review_Package.zip")
uploaded = files.upload()
matches = [
    content for content in uploaded.values()
    if hashlib.sha256(content).hexdigest() == EXPECTED_SHA256
]

if len(matches) != 1:
    raise ValueError("Upload the linked package ZIP unchanged.")

remediation_package_bytes = matches[0]
print("Replacement-package checksum verified.")

with zipfile.ZipFile(io.BytesIO(remediation_package_bytes)) as bundle:
    review_code = bundle.read("block_06G.py")

exec(
    compile(review_code, "Task_2_Block_06G_Replacement_Review.py", "exec"),
    globals(),
)

In [ ]:
from google.colab import files
import hashlib, io, zipfile

uploaded_6h = files.upload()
if len(uploaded_6h) != 1:
    raise ValueError("Upload only Task_2_Block_06H_Review_Package.zip")

remediation_package_bytes = bytes(next(iter(uploaded_6h.values())))
expected_6h = "26003d840ab00d6f652051c05a548a195ba2636aeacca857d0aa62ceec5ea11a"
if hashlib.sha256(remediation_package_bytes).hexdigest() != expected_6h:
    raise ValueError("Package checksum mismatch.")

with zipfile.ZipFile(io.BytesIO(remediation_package_bytes)) as package_6h:
    if package_6h.testzip() is not None:
        raise ValueError("ZIP integrity check failed.")
    code_6h = package_6h.read("block_06H.py")

if hashlib.sha256(code_6h).hexdigest() != "05ca1aeec1316027b77abfdbbba07bfc39e71bcdc02568c4998ce02d258592f9":
    raise ValueError("Review code checksum mismatch.")

exec(compile(code_6h, "block_06H.py", "exec"), globals())

In [ ]:
from google.colab import files
import hashlib, io, zipfile

uploaded_6i = files.upload()
if len(uploaded_6i) != 1:
    raise ValueError("Upload only Task_2_Block_06I_Review_Package.zip")
remediation_package_bytes = bytes(next(iter(uploaded_6i.values())))
if hashlib.sha256(remediation_package_bytes).hexdigest() != "80f330de2846e7498738ff6dc0a782bfa9c5eaddb1001402323092976995214e":
    raise ValueError("Package checksum mismatch.")
with zipfile.ZipFile(io.BytesIO(remediation_package_bytes)) as package_6i:
    if package_6i.testzip() is not None:
        raise ValueError("ZIP integrity check failed.")
    code_6i = package_6i.read("block_06I.py")
if hashlib.sha256(code_6i).hexdigest() != "051d1d42e0a45697557c2e4ff696e06871f6942821143a3d2484f09f70d5643d":
    raise ValueError("Review code checksum mismatch.")
exec(compile(code_6i, "block_06I.py", "exec"), globals())

In [ ]:
from google.colab import files
import hashlib, io, zipfile

uploaded_6j = files.upload()
if len(uploaded_6j) != 1:
    raise ValueError("Upload only Task_2_Block_06J_Review_Package.zip")
remediation_package_bytes = bytes(next(iter(uploaded_6j.values())))
if hashlib.sha256(remediation_package_bytes).hexdigest() != "ede51261a020b687286a60e00397f37310cfb145c89f7c772b8c600a218e868a":
    raise ValueError("Package checksum mismatch.")
with zipfile.ZipFile(io.BytesIO(remediation_package_bytes)) as package_6j:
    if package_6j.testzip() is not None:
        raise ValueError("ZIP integrity check failed.")
    code_6j = package_6j.read("block_06J.py")
if hashlib.sha256(code_6j).hexdigest() != "414e0685cac02b49d4435b0c5e838c64a4c487ddff2d61d61588da23242a5a34":
    raise ValueError("Review code checksum mismatch.")
exec(compile(code_6j, "block_06J.py", "exec"), globals())


In [ ]:
from google.colab import files
import hashlib, io, zipfile

uploaded_6k = files.upload()
if len(uploaded_6k) != 1:
    raise ValueError("Upload only Task_2_Block_06K_Review_Package.zip")
remediation_package_bytes = bytes(next(iter(uploaded_6k.values())))
if hashlib.sha256(remediation_package_bytes).hexdigest() != "d97d6fa55b9e930fbcced1676b7090455f5eb93d1a547bfeaedb7ddd7db4748b":
    raise ValueError("Package checksum mismatch.")
with zipfile.ZipFile(io.BytesIO(remediation_package_bytes)) as package_6k:
    if package_6k.testzip() is not None:
        raise ValueError("ZIP integrity check failed.")
    code_6k = package_6k.read("block_06K.py")
if hashlib.sha256(code_6k).hexdigest() != "1a1968ab691ecd1a7a689e136e7066c40fd62ff5b6e963fae93c60695a0fb39b":
    raise ValueError("Review code checksum mismatch.")
exec(compile(code_6k, "block_06K.py", "exec"), globals())

In [ ]:
from google.colab import files
import hashlib, io, zipfile

uploaded_07 = files.upload()
if len(uploaded_07) != 1:
    raise ValueError("Upload only Task_2_Block_07_CPU_Preflight.zip")
task2_preflight_package_bytes = bytes(next(iter(uploaded_07.values())))
if hashlib.sha256(task2_preflight_package_bytes).hexdigest() != "d5d2ad21658d452673d6a549f725e2f78949b97b3890ee9d93e170d6eb50ec95":
    raise ValueError("Package checksum mismatch.")
with zipfile.ZipFile(io.BytesIO(task2_preflight_package_bytes)) as package_07:
    if package_07.testzip() is not None:
        raise ValueError("ZIP integrity check failed.")
    code_07 = package_07.read("block_07.py")
if hashlib.sha256(code_07).hexdigest() != "7bc799b420e2e0290da306a894cd512208972d976306eca827b5f54e8473b309":
    raise ValueError("Preflight code checksum mismatch.")
exec(compile(code_07, "block_07.py", "exec"), globals())

In [ ]:
from google.colab import files
import hashlib, io, zipfile

uploaded_08 = files.upload()
if len(uploaded_08) != 1:
    raise ValueError("Upload only Task_2_Block_08_Image_Preparation.zip")
task2_image_package_bytes = bytes(next(iter(uploaded_08.values())))
if hashlib.sha256(task2_image_package_bytes).hexdigest() != "3882b1174fe015da1dc4b9de7d97abc07038a120ec49d8d21053c865bb109160":
    raise ValueError("Package checksum mismatch.")
with zipfile.ZipFile(io.BytesIO(task2_image_package_bytes)) as package_08:
    if package_08.testzip() is not None:
        raise ValueError("ZIP integrity check failed.")
    code_08 = package_08.read("block_08.py")
if hashlib.sha256(code_08).hexdigest() != "9b036fe962d183dc204c8991b2b77f90495d5c8bc614716287ca0f773b902f60":
    raise ValueError("Image-preparation code checksum mismatch.")
exec(compile(code_08, "block_08.py", "exec"), globals())

In [ ]:
from google.colab import files
import hashlib, io, zipfile

uploaded_09 = files.upload()
if len(uploaded_09) != 1:
    raise ValueError("Upload only Task_2_Block_09_Cache_Input_Packing.zip")

task2_staging_package_bytes = bytes(next(iter(uploaded_09.values())))
if hashlib.sha256(task2_staging_package_bytes).hexdigest() != "571c75773e78e3cec1e3a0c32ff8f8e7a30a3cd3dd63dbb745e56bd667155027":
    raise ValueError("Package checksum mismatch.")

with zipfile.ZipFile(io.BytesIO(task2_staging_package_bytes)) as package_09:
    if package_09.testzip() is not None:
        raise ValueError("ZIP integrity check failed.")
    code_09 = package_09.read("block_09.py")

if hashlib.sha256(code_09).hexdigest() != "282ca2935b66f3741ea006a085b83f29c3b2d42ab625f50605d4c9c3be018c0b":
    raise ValueError("Packing code checksum mismatch.")

exec(compile(code_09, "block_09.py", "exec"), globals())

In [ ]:
from google.colab import files
import hashlib, io, zipfile

uploaded_10 = files.upload()
if len(uploaded_10) != 1:
    raise ValueError("Upload only Task_2_Block_10_Model_Preparation.zip")

task2_models_package_bytes = bytes(next(iter(uploaded_10.values())))
if hashlib.sha256(task2_models_package_bytes).hexdigest() != "41ceb7a820130a0d886d8391b46bdb7ddef1e537ca05256ae28b0793127d5288":
    raise ValueError("Package checksum mismatch.")

with zipfile.ZipFile(io.BytesIO(task2_models_package_bytes)) as package_10:
    if package_10.testzip() is not None:
        raise ValueError("ZIP integrity check failed.")
    code_10 = package_10.read("block_10.py")

if hashlib.sha256(code_10).hexdigest() != "d662bef8b31e070532dbfc3df3afd6ae9f206a973e64fbae21cd9fcea6385187":
    raise ValueError("Model-preparation code checksum mismatch.")

exec(compile(code_10, "block_10.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded_11 = files.upload()
    if len(uploaded_11) != 1:
        raise ValueError("Upload only Task_2_Block_11_GPU_Feature_Cache.zip")

    task2_features_package_bytes = bytes(next(iter(uploaded_11.values())))
    if hashlib.sha256(task2_features_package_bytes).hexdigest() != "44ba8a4a77df2234578b53109be7033861a0c1f128a9d1ea7cc55014bc066a6e":
        raise ValueError("Package checksum mismatch.")

    with zipfile.ZipFile(io.BytesIO(task2_features_package_bytes)) as package_11:
        if package_11.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code_11 = package_11.read("block_11.py")

    if hashlib.sha256(code_11).hexdigest() != "7b59c339e656db0b3cbab4f5a5dcbd055c5e17483945ce767939c9109a91f007":
        raise ValueError("Cache-controller checksum mismatch.")

except BaseException:
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code_11, "block_11.py", "exec"), globals())

In [ ]:
import os, json
from pathlib import Path
from google.colab import drive, runtime

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    try:
        raise RuntimeError("Select CPU, reconnect, then run this cell.")
    finally:
        runtime.unassign()

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

root = Path("/content/drive/MyDrive/ATML_PA3/task2/run_20261031_150949_1cff148a")
dataset = root / "datasets/task2_A012_82ad5006bd1d68ee"
logs = root / "logs"

print("BLOCK 11 CPU DIAGNOSTIC")
found = set()

for pattern in ("block_11_failed_*.json", "block_11_*.json"):
    for path in sorted(logs.glob(pattern), reverse=True)[:3]:
        if path.name not in found:
            found.add(path.name)
            print("\nSAVED LOG", path.name)
            print(path.read_text())

if not found:
    print("No Block 11 result or failure JSON was saved.")

features = dataset / "features"
binding = features / "siglip196_recipe.json"
print("\nFeature recipe exists", binding.is_file())

if binding.is_file():
    recipe = json.loads(binding.read_text())
    print("Saved environment", json.dumps(recipe.get("environment", {}), indent=2))

for folder in sorted(features.glob("siglip196_*")):
    if folder.is_dir():
        print("Cache inventory", folder.name,
              "receipts", len(list(folder.glob("shard_*.json"))),
              "arrays", len(list(folder.glob("shard_*.npy"))),
              "completion", (folder / "features_complete.json").is_file())

pointer = root / "manifests/active_task2_features.json"
print("\nActive feature pointer exists", pointer.is_file())
if pointer.is_file():
    print(pointer.read_text())

print("\nCPU diagnostic finished.")

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded_11r = files.upload()
    if len(uploaded_11r) != 1:
        raise ValueError("Upload only Task_2_Block_11R1_GPU_Feature_Cache.zip")

    task2_features_package_bytes = bytes(next(iter(uploaded_11r.values())))
    if hashlib.sha256(task2_features_package_bytes).hexdigest() != "fca343eded123a3003a63cfeca60dbb090e6ba29a0cba5a82a769f92529ef536":
        raise ValueError("Choose the corrected 11R1 ZIP.")

    with zipfile.ZipFile(io.BytesIO(task2_features_package_bytes)) as package_11r:
        if package_11r.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code_11r = package_11r.read("block_11.py")

    if hashlib.sha256(code_11r).hexdigest() != "7a769c130487d02620fea3ca7595fbca34cda17ae17776338ecb71d10f64df10":
        raise ValueError("Cache-controller checksum mismatch.")

except BaseException:
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code_11r, "block_11.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded_12 = files.upload()
    if len(uploaded_12) != 1:
        raise ValueError("Upload only Task_2_Block_12_CPU_Text_Preparation.zip")

    task2_text_package_bytes = bytes(next(iter(uploaded_12.values())))
    if hashlib.sha256(task2_text_package_bytes).hexdigest() != "fe3b827e8ad6367c661a5846711eef84eb31dea472484bec4dd3d0e717da463b":
        raise ValueError("Package checksum mismatch.")

    with zipfile.ZipFile(io.BytesIO(task2_text_package_bytes)) as package_12:
        if package_12.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code_12 = package_12.read("block_12.py")

    if hashlib.sha256(code_12).hexdigest() != "7e28b5ac6035ec7c8363f2b931b051b4874e9187c4c95acebdfbc81df78ffba4":
        raise ValueError("CPU controller checksum mismatch.")

except BaseException:
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code_12, "block_12.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded_13 = files.upload()
    if len(uploaded_13) != 1:
        raise ValueError("Upload only Task_2_Block_13_GPU_Initialization.zip")

    task2_initialization_package_bytes = bytes(
        next(iter(uploaded_13.values()))
    )
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "636cee5840e575871b4db51ae158c2eede4dc22f8f1b24637849d0911b9a7fb9":
        raise ValueError("Package checksum mismatch.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package_13:
        if package_13.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code_13 = package_13.read("block_13.py")

    if hashlib.sha256(code_13).hexdigest() != "5148e43452c0cfc55edc698980cd4be3b6c46860c46a22aed81a6029bc4ad02b":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code_13, "block_13.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded_13 = files.upload()
    if len(uploaded_13) != 1:
        raise ValueError("Upload only Task_2_Block_13_GPU_Initialization.zip")

    task2_initialization_package_bytes = bytes(
        next(iter(uploaded_13.values()))
    )
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "636cee5840e575871b4db51ae158c2eede4dc22f8f1b24637849d0911b9a7fb9":
        raise ValueError("Package checksum mismatch.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package_13:
        if package_13.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code_13 = package_13.read("block_13.py")

    if hashlib.sha256(code_13).hexdigest() != "5148e43452c0cfc55edc698980cd4be3b6c46860c46a22aed81a6029bc4ad02b":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code_13, "block_13.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_13R1_GPU_Initialization.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "0833b78d59608d162fc66e2f321b8b3b4335d99f219be1f157e411ac003a86ed":
        raise ValueError("Wrong ZIP. Upload the new 13R1 package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_13.py")

    if hashlib.sha256(code).hexdigest() != "5148e43452c0cfc55edc698980cd4be3b6c46860c46a22aed81a6029bc4ad02b":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_13.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_13D_GPU_Gradient_Diagnostic.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "593c67f036a8aafe784b372cb07ee55c20126d00221c337ec5dbebebd970a7f1":
        raise ValueError("Wrong ZIP. Upload the new 13D diagnostic package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_13D.py")

    if hashlib.sha256(code).hexdigest() != "9127366b3050587815b5bcca768dcd58cc9239796f69049667f227cea471bfa3":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_13D.py", "exec"), globals())


In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_13R2_GPU_Initialization.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "8fc0a3270bb36adec661872d34a5af5703b403b32c95896851e8339f04827db3":
        raise ValueError("Wrong ZIP. Upload the approved 13R2 AMP128 package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_13.py")

    if hashlib.sha256(code).hexdigest() != "5148e43452c0cfc55edc698980cd4be3b6c46860c46a22aed81a6029bc4ad02b":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_13.py", "exec"), globals())


In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "bbd8f6986d4fbd8da50cd2e0c384075f5bc5cb709712ebbae9db40f8df24fcf1":
        raise ValueError("Wrong ZIP. Upload Task_2_Block_14_GPU_Training.zip")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "e661c09f2c0b085ba76db8bb9a9ebae3985ae65a6a2898c65b781c21b5b1ef0c":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "bbd8f6986d4fbd8da50cd2e0c384075f5bc5cb709712ebbae9db40f8df24fcf1":
        raise ValueError("Wrong ZIP. Upload Task_2_Block_14_GPU_Training.zip")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "e661c09f2c0b085ba76db8bb9a9ebae3985ae65a6a2898c65b781c21b5b1ef0c":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_14R1_GPU_Training.zip")

    task2_initialization_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_initialization_package_bytes).hexdigest() != "144fd153c443dc57e30d1cf4e7236ef3b91995c610d90a583dd09c62530165ed":
        raise ValueError("Wrong ZIP. Upload the new 14R1 training package.")

    with zipfile.ZipFile(io.BytesIO(task2_initialization_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_14.py")

    if hashlib.sha256(code).hexdigest() != "b40c078d7aa3ef33cc7ab01082d12a69ea953390e321074a0bc9fc0af7706c52":
        raise ValueError("Controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed; do not continue.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_14.py", "exec"), globals())

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, re

if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
    raise RuntimeError("Switch Hardware accelerator to CPU first.")
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

root = Path("/content/drive/MyDrive/ATML_PA3/task2/run_20261031_150949_1cff148a")
logs = root / "logs"
print("BLOCK 14 RECOVERY CHECK — NO TRAINING")

if not logs.is_dir():
    raise FileNotFoundError("Run logs unavailable. Check the mounted Google account.")

paths = list(logs.glob("block_14_*"))
attempts = sorted({
    m.group(0) for p in paths
    for m in [re.search(r"\d{8}_\d{6}_[0-9a-f]{8}", p.name)] if m
})

for attempt in attempts[-2:]:
    print("\nATTEMPT:", attempt)
    for prefix in ("block_14_", "block_14_failed_",
                   "block_14_controller_", "block_14_status_"):
        p = logs / (prefix + attempt + ".json")
        print("\n" + p.name)
        print(p.read_text() if p.exists() else "NOT FOUND")

if attempts:
    attempt = attempts[-1]
    phases = sorted(logs.glob("block_14_phase_" + attempt + "_*.json"))
    if phases:
        print("\nLATEST SAVED PHASE\n" + phases[-1].read_text())
    consoles = list(logs.glob("block_14_console_" + attempt + "_*.log"))
    if consoles:
        p = max(consoles, key=lambda p: (p.stat().st_mtime_ns, p.name))
        print("\nLATEST SAVED CONSOLE TAIL:", p.name)
        with p.open("rb") as stream:
            stream.seek(max(0, p.stat().st_size - 18000))
            print(stream.read().decode("utf-8", errors="replace"))

pointer_path = root / "manifests/active_task2_training.json"
print("\nACTIVE TRAINING POINTER")
if pointer_path.exists():
    raw_pointer = pointer_path.read_bytes()
    pointer = json.loads(raw_pointer)
    print(json.dumps(pointer, indent=2))

    receipt = pointer["progress"]
    p = Path(receipt["path"])
    if not p.resolve().is_relative_to(root.resolve()):
        raise ValueError("Progress path is outside the task run.")
    raw = p.read_bytes()
    if (len(raw) != receipt["bytes"] or
        hashlib.sha256(raw).hexdigest() != receipt["sha256"]):
        raise ValueError("Saved progress receipt does not match.")

    progress = json.loads(raw)
    checkpoint = progress["checkpoint"]
    p = Path(checkpoint["path"])
    if not p.resolve().is_relative_to(root.resolve()):
        raise ValueError("Checkpoint path is outside the task run.")
    digest = hashlib.sha256()
    with p.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    if (p.stat().st_size != checkpoint["bytes"] or
        digest.hexdigest() != checkpoint["sha256"]):
        raise ValueError("Saved checkpoint does not match its receipt.")
    if (progress["global_updates"] != pointer["global_updates"] or
        pointer_path.read_bytes() != raw_pointer):
        raise ValueError("Progress changed or counters disagree.")

    print("CHECKPOINT FILE HASH VERIFIED")
    print("SAVED UPDATE:", progress["global_updates"])
    print("STAGE COUNTS:", progress["stage_updates"])
else:
    print("NOT FOUND")

print("\nDIAGNOSTIC FINISHED — send the entire output before resuming training.")

In [ ]:
from google.colab import files, runtime
import hashlib, io, os, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_15_GPU_Final_Evaluation.zip")

    task2_evaluation_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_evaluation_package_bytes).hexdigest() != "e5f2890a6fac6dd70162d5e04e3b53157b3a8c612d4ae94436090f3831a2c5ac":
        raise ValueError("Wrong or damaged ZIP. Upload the Block 15 package.")

    with zipfile.ZipFile(io.BytesIO(task2_evaluation_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_15.py")

    if hashlib.sha256(code).hexdigest() != "47f583783062473e8847e848b3a6c2d656329d2d68048e4c402676f5d707958e":
        raise ValueError("Evaluation controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed.", flush=True)
    if os.environ.get("COLAB_GPU", "0") not in ("", "0"):
        runtime.unassign()
    raise

exec(compile(code, "block_15.py", "exec"), globals())

In [ ]:
from google.colab import files
import hashlib, io, zipfile

try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only Task_2_Block_16_CPU_Results_and_Plots.zip")

    task2_results_package_bytes = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(task2_results_package_bytes).hexdigest() != "5378a536e1aca6a982f57e8ddeeb7a25d8f35d8a0546e7f640f1d44439c971bd":
        raise ValueError("Wrong or damaged ZIP. Upload the Block 16 package.")

    with zipfile.ZipFile(io.BytesIO(task2_results_package_bytes)) as package:
        if package.testzip() is not None:
            raise ValueError("ZIP integrity check failed.")
        code = package.read("block_16.py")

    if hashlib.sha256(code).hexdigest() != "3f9acf3f46993aab926edd533574ed4fd40cb3e5edeeb31fe8f4a2d4193af732":
        raise ValueError("CPU results controller checksum mismatch.")

except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Package loading failed.", flush=True)
    raise

try:
    exec(compile(code, "block_16.py", "exec"), globals())
except BaseException:
    print("STOP - SEND THIS OUTPUT TO ME. Keep the full output for review.", flush=True)
    raise